In [0]:
##aws_utils
import boto3
from botocore.exceptions import NoCredentialsError
 
from pyspark.dbutils import DBUtils
from pyspark.sql import SparkSession

from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from datetime import datetime
 
spark = SparkSession.builder.getOrCreate()
dbutils = DBUtils(spark)
 
# Function to get AWS credentials from Databricks secrets
def get_aws_credentials(secrets_scope, key_name):
    try:  
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
 
# Function to get AWS credentials from Databricks secrets
def get_scope_key_names(secrets_scope):
    try:
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
   
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    
 
# Function to connect to AWS and access SSM parameter
def connect_to_aws_and_access_ssm(ssm_key, ssm_client, aws_secret_key):
 
    if aws_secret_key is None:
        print("AWS secret key not available.")
        return
 
    # Access SSM parameter /cfn/a360/reports_output_path
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )
 
    return response['Parameter']['Value']
 
# Function to get the environment prefix
def get_env_prefix():
    secrets_scope = "aws-databricks"
    aws_keys = get_scope_key_names(secrets_scope)
    aws_key = aws_keys[0] if aws_keys else None
    # Get AWS secret key from Databricks secrets
    aws_secret_key = get_aws_credentials(secrets_scope,aws_key)
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                            aws_access_key_id=aws_key,
                            aws_secret_access_key=aws_secret_key)
 
    S3_client = boto3.client('s3', region_name='eu-central-1',
                            aws_access_key_id=aws_key,
                            aws_secret_access_key=aws_secret_key)  
 
    env = connect_to_aws_and_access_ssm('/cfn/bucket_prefix', ssm_client, aws_secret_key)
    return env

env=get_env_prefix()



widgets = dbutils.widgets.getAll()

run_date = widgets.get("run_date")

if not run_date or not run_date.strip():
    run_date = spark.sql(f"""
        SELECT max(BusinessDate) as max_date
        FROM `{env}-cdh`.primaryactive.insurance
    """).first()["max_date"]

run_date = str(run_date)

print(f"Run date: {run_date}")

In [0]:
# env = 'p'
# run_date = '2026-02-26'

In [0]:
insurance = spark.sql(f"""
       SELECT DISTINCT AgreementNumber,
                     InsuranceStartDate,
                     InsuranceEndDate,
                     GroupInsuranceflag,
                     isactiveflag,
                     action,
                     NetInsuranceAmount,
                     ForecastInsuranceAmount,
                     InsurancePool,
                     InsuranceReceivedDate,
                     InsuranceEnteredDate,
                     realestatekey,
                     insurancekey,
                     companykey
       FROM `{env}-cdh`.primaryactive.insurance
       WHERE BusinessDate = {run_date!r}
""")

insurance.createOrReplaceTempView("insurance")


thesis_pr_insurance = spark.sql(f""" 
       SELECT DISTINCT id,
                     master_id
       FROM `{env}-cdh`.landed.thesis_prinsurance
       WHERE BusinessDate = {run_date!r}
""")

thesis_pr_insurance.createOrReplaceTempView("thesis_pr_insurance")


realestatebuilding = spark.sql(f"""
       SELECT DISTINCT RealEstateCepalKey,
                     realestateactive,
                     RealEstateKey
       FROM `{env}-cdh`.primaryactive.realestatebuilding
       WHERE BusinessDate = {run_date!r}
""")

realestatebuilding.createOrReplaceTempView("realestatebuilding")


realestateinfo = spark.sql(f"""
       SELECT DISTINCT RealEstateKey
       FROM `{env}-cdh`.primaryactive.realestateinfo
       WHERE BusinessDate = {run_date!r}
""")

realestateinfo.createOrReplaceTempView("realestateinfo")


collateral = spark.sql(f"""
       SELECT DISTINCT collateralstatus,
                     AgreementKey,
                     CollateralKey, 
                     RealEstateKey
       FROM `{env}-cdh`.primaryactive.collateral
       WHERE BusinessDate = {run_date!r}
""")

collateral.createOrReplaceTempView("collateral")


agreement = spark.sql(f"""
       SELECT DISTINCT PortfolioName,
                     AgreementKey 
       FROM `{env}-cdh`.primaryactive.agreement
       WHERE BusinessDate = {run_date!r}
""")

agreement.createOrReplaceTempView("agreement")


dexposure = spark.sql(f"""
       SELECT  DISTINCT AgreementKey,
                     CDI,
                     SourceSystem,
                     slaveFlag,
                     ExposureKey,
                     ProductKey,
                     IsSubsidyFlag,
                     OutstandingTotalBalance,
                     NextInstallmentAmount,
                     ExposureStatusCode,
                     ExposureStatus,
                     PartyName,
                     CaseCode,
                     PortfolioKey,
                     GBV,
                     DenouncedDate,
                     DaysPastDue,
                     PEERestructureTypeCode,
                     HasActiveSettlementFlag,
                     NextInstallmentDate,
                     NextInstallmentAmount,
                     BusinessDate
       FROM `p-cdh`.derivative.get_all_exposures_by_business_date(DATE({run_date!r})) --FIX!!!!!
""")

dexposure.createOrReplaceTempView("dexposure")


dexposure_profits = spark.sql(f"""
       SELECT DISTINCT AgreementKey,
                     CDI,
                     SourceSystem,
                     slaveFlag,
                     ExposureKey,
                     ProductKey,
                     IsSubsidyFlag,
                     OutstandingTotalBalance,
                     NextInstallmentAmount,
                     ExposureStatusCode,
                     ExposureStatus,
                     PartyName,
                     CaseCode,
                     PortfolioKey,
                     GBV,
                     DenouncedDate,
                     DaysPastDue,
                     PEERestructureTypeCode,
                     HasActiveSettlementFlag,
                     NextInstallmentDate,
                     NextInstallmentAmount,
                     BusinessDate
       FROM `p-cdh`.derivative.get_all_exposures_by_business_date(DATE({run_date!r})) --FIXED!!!!
       WHERE SourceSystem = 'profits'
""")

dexposure_profits.createOrReplaceTempView("dexposure_profits")


party = spark.sql(f"""
       SELECT DISTINCT PartyKey,
                     PartyType,
                     CDI,
                     LastNameGR,
                     PartyRepresentKey
       FROM `{env}-cdh`.primaryactive.party
       WHERE BusinessDate = {run_date!r}
""")

party.createOrReplaceTempView("party")


dparty = spark.sql(f"""
       SELECT DISTINCT CDI, 
                     PrimaryCommercialGR,
                     TaxRegistrationNumber,
                     GroupCode,
                     GroupName,
                     PartyKey
       FROM `{env}-cdh`.derivativeactive.dparty
       WHERE BusinessDate = {run_date!r}
""")

dparty.createOrReplaceTempView("dparty")


portfolio = spark.sql(f"""
       SELECT DISTINCT Prefix,
                     PortfolioName,
                     unit_code_profits,
                     ParentPortfolio
       FROM `{env}-cdh`.primaryactive.portfolio
       WHERE BusinessDate = {run_date!r}
""")

portfolio.createOrReplaceTempView("portfolio")


product = spark.sql(f"""
       SELECT DISTINCT ProductKey,
                     ProductDescription,
                     Subproduct 
       FROM `{env}-cdh`.primaryactive.product
       WHERE BusinessDate = {run_date!r}
""")

product.createOrReplaceTempView("product")


dpaymentschedule = spark.sql(f"""
       SELECT DISTINCT InstallmentDate,
                     InstallmentAmount,
                     InstallmentStatus,
                     ExposureKey,
                     BusinessDate 
       FROM `{env}-cdh`.derivativeactive.dpaymentschedule
       WHERE BusinessDate = {run_date!r}
       AND SourceSystem = "profits"
""")

dpaymentschedule.createOrReplaceTempView("dpaymentschedule")


UnderLaw = spark.sql(f"""
       SELECT DISTINCT CDI,
                     InLawFlag,
                     UnderLawTypeCode
       FROM `{env}-cdh`.primaryactive.underlaw
       WHERE BusinessDate = {run_date!r}
""")

UnderLaw.createOrReplaceTempView("UnderLaw")


casecodeview = spark.sql(f"""
       SELECT DISTINCT CaseCodeKey,
                     TransactionPerimeterCode,
                     sourcesystem
       FROM `{env}-cdh`.primaryactive.casecode 
       WHERE BusinessDate = {run_date!r}
""")

casecodeview.createOrReplaceTempView("casecodeview")


contactdetails = spark.sql(f"""
       SELECT DISTINCT Street,
                     StreetNumber,
                     ContactDetailType,
                     CommunicationFlag,
                     PartyKey,
                     County,
                     PostalCode,
                     CDI
       FROM `{env}-cdh`.primaryactive.contactdetails 
       WHERE BusinessDate = {run_date!r} 
""")

contactdetails.createOrReplaceTempView("contactdetails")


geographicalarea = spark.sql(f"""
       SELECT DISTINCT County,
                     PostalCode
       FROM `{env}-cdh`.reference.geographicalarea
""")

geographicalarea.createOrReplaceTempView("geographicalarea")


danalyticsmetrics = spark.sql(f"""
       SELECT DISTINCT ExposureKey,
                     PrimaryPartyCDI,
                     PayerFlagEntity
       FROM `{env}-cdh`.derivativeactive.danalyticsmetrics
       WHERE BusinessDate = {run_date!r} 
""")

danalyticsmetrics.createOrReplaceTempView("danalyticsmetrics")


settlementexposure = spark.sql(f"""
       SELECT DISTINCT ExposureKey,
                     SettlementStatus
       FROM `{env}-cdh`.primaryactive.settlementexposure
       WHERE BusinessDate = {run_date!r} 
""")

settlementexposure.createOrReplaceTempView("settlementexposure")


In [0]:
# # -- ============================================================
# # -- ts (Base Insurance Perimeter)
# # --
# # -- This perimeter represents the **base / original insurance
# # -- contracts**.
# # --
# # -- It includes:
# # -- - Standalone insurance contracts
# # -- - The parent (master) insurance records that additive
# # --   actions may later refer to
# # --
# # -- Each record in ts acts as an **anchor insurance contract**
# # -- and defines the initial insurance coverage per property
# # -- and agreement.
# # --
# # -- This perimeter is later used to:
# # -- - Link additive insurance actions
# # -- - Determine the correct insurance contract reference
# # -- - Avoid double counting insurance coverage
# # -- After the procedure's release on prod this table, and all the tables that have been used to save time will be dropped
# # -- ============================================================

base_insurance_df = (
    spark.table("insurance")
    .join(
        spark.table("thesis_pr_insurance"),
        F.col("thesis_pr_insurance.id").cast("string") == F.trim(F.split(F.col("insurance.insurancekey").cast("string"), ";").getItem(0)),
        "left",
    )
    .join(
        spark.table("realestatebuilding"),
        F.col("insurance.realestatekey").cast("string") == F.trim(F.col("realestatebuilding.RealEstateKey").cast("string")),
        "inner",
    )
    .join(
        spark.table("realestateinfo"),
        F.col("insurance.realestatekey").cast("string") == F.trim(F.col("realestateinfo.RealEstateKey").cast("string")),
        "inner",
    )
    .join(
        spark.table("collateral"),
        F.col("insurance.realestatekey").cast("string") == F.trim(F.col("collateral.RealEstateKey").cast("string")),
        "left",
    )
    .join(
        spark.table("dexposure"),
        F.col("dexposure.agreementkey").cast("string") == F.trim(F.col("collateral.agreementkey").cast("string")),
        "left",
    )
    .join(
        spark.table("party"),
        (F.col("insurance.companykey").cast("string") == F.trim(F.col("party.PartyKey").cast("string"))) & 
        (F.col("party.PartyType") == "Organization") & 
        (F.col("party.PartyRepresentKey").like("%INSUR%")),
        "left",
    )
    .join(
        spark.table("dparty"),
        F.col("dparty.CDI").cast("string") == F.trim(F.col("dexposure.CDI").cast("string")),
        "left",
    )
    .join(
        spark.table("agreement"),
        F.col("agreement.AgreementKey").cast("string") == F.trim(F.col("collateral.AgreementKey").cast("string")),
        "left"
    )
    .join(
        spark.table("portfolio"),
        F.col("agreement.PortfolioName").cast("string")  == F.trim(F.col("portfolio.PortfolioName").cast("string")),
        "left"
    )
    .select(
        F.col("thesis_pr_insurance.id").alias("ThesisId"),
        F.col("thesis_pr_insurance.master_id").alias("MasterId"),
        F.col("insurance.AgreementNumber"),
        F.col("InsuranceStartDate"),
        F.col("InsuranceEndDate"),
        F.col("GroupInsuranceflag"),
        F.col("isactiveflag"),
        F.col("action"),
        F.col("NetInsuranceAmount"),
        F.col("ForecastInsuranceAmount"),
        F.col("InsurancePool"),
        F.col("InsuranceReceivedDate"),
        F.col("InsuranceEnteredDate"),
        F.col("insurance.realestatekey"),
        F.col("realestatebuilding.RealEstateCepalKey").alias("cepal_key"),
        F.col("realestatebuilding.realestateactive"),
        F.col("collateral.collateralstatus"),
        F.col("collateral.AgreementKey"),
        F.col("collateral.CollateralKey"),
        F.col("party.LastNameGR").alias("InsuranceCompany"),
        F.col("dparty.PrimaryCommercialGR"),
        F.col("dparty.TaxRegistrationNumber"),
        F.col("dparty.CDI"),
        F.col("agreement.PortfolioName"),
        F.col("portfolio.Prefix")
    )
    .dropDuplicates()
)

base_insurance_df = base_insurance_df.withColumn(
    "BusinessDate", F.lit(run_date).cast("date")
)

base_insurance_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"`{env}-cdh-delivery`.internalreports.ip_base_insurance")
spark.sql(f"OPTIMIZE `{env}-cdh-delivery`.internalreports.ip_base_insurance")
base_insurance_df = spark.table(f"`{env}-cdh-delivery`.internalreports.ip_base_insurance")

In [0]:
# # -- ============================================================
# # -- ts (Base Insurance Perimeter)
# # --
# # -- This perimeter represents the **base / original insurance
# # -- contracts**.
# # --
# # -- It includes:
# # -- - Standalone insurance contracts
# # -- - The parent (master) insurance records that additive
# # --   actions may later refer to
# # --
# # -- Each record in ts acts as an **anchor insurance contract**
# # -- and defines the initial insurance coverage per property
# # -- and agreement.
# # --
# # -- This perimeter is later used to:
# # -- - Link additive insurance actions
# # -- - Determine the correct insurance contract reference
# # -- - Avoid double counting insurance coverage
# # -- After the procedure's release on prod this table, and all the tables that have been used to save time will be dropped
# # -- ============================================================

base_insurance_df = (
    spark.table("insurance")
    .join(
        spark.table("thesis_pr_insurance"),
        F.col("thesis_pr_insurance.id").cast("string") == F.trim(F.split(F.col("insurance.insurancekey").cast("string"), ";").getItem(0)),
        "left",
    )
    .join(
        spark.table("realestatebuilding"),
        F.col("insurance.realestatekey").cast("string") == F.trim(F.col("realestatebuilding.RealEstateKey").cast("string")),
        "inner",
    )
    .join(
        spark.table("realestateinfo"),
        F.col("insurance.realestatekey").cast("string") == F.trim(F.col("realestateinfo.RealEstateKey").cast("string")),
        "inner",
    )
    .join(
        spark.table("collateral"),
        F.col("insurance.realestatekey").cast("string") == F.trim(F.col("collateral.RealEstateKey").cast("string")),
        "left",
    )
    .join(
        spark.table("dexposure"),
        F.col("dexposure.agreementkey").cast("string") == F.trim(F.col("collateral.agreementkey").cast("string")),
        "left",
    )
    .join(
        spark.table("party"),
        (F.col("insurance.companykey").cast("string") == F.trim(F.col("party.PartyKey").cast("string"))) & 
        (F.col("party.PartyType") == "Organization") & 
        (F.col("party.PartyRepresentKey").like("%INSUR%")),
        "left",
    )
    .join(
        spark.table("dparty"),
        F.col("dparty.CDI").cast("string") == F.trim(F.col("dexposure.CDI").cast("string")),
        "left",
    )
    .join(
        spark.table("agreement"),
        F.col("agreement.AgreementKey").cast("string") == F.trim(F.col("collateral.AgreementKey").cast("string")),
        "left"
    )
    .join(
        spark.table("portfolio"),
        F.col("agreement.PortfolioName").cast("string")  == F.trim(F.col("portfolio.PortfolioName").cast("string")),
        "left"
    )
    .select(
        F.col("thesis_pr_insurance.id").alias("ThesisId"),
        F.col("thesis_pr_insurance.master_id").alias("MasterId"),
        F.col("insurance.AgreementNumber"),
        F.col("InsuranceStartDate"),
        F.col("InsuranceEndDate"),
        F.col("GroupInsuranceflag"),
        F.col("isactiveflag"),
        F.col("action"),
        F.col("NetInsuranceAmount"),
        F.col("ForecastInsuranceAmount"),
        F.col("InsurancePool"),
        F.col("InsuranceReceivedDate"),
        F.col("InsuranceEnteredDate"),
        F.col("insurance.realestatekey"),
        F.col("realestatebuilding.RealEstateCepalKey").alias("cepal_key"),
        F.col("realestatebuilding.realestateactive"),
        F.col("collateral.collateralstatus"),
        F.col("collateral.AgreementKey"),
        F.col("collateral.CollateralKey"),
        F.col("party.LastNameGR").alias("InsuranceCompany"),
        F.col("dparty.PrimaryCommercialGR"),
        F.col("dparty.TaxRegistrationNumber"),
        F.col("dparty.CDI"),
        F.col("agreement.PortfolioName"),
        F.col("portfolio.Prefix")
    )
    .dropDuplicates()
)

base_insurance_df = base_insurance_df.withColumn(
    "BusinessDate", F.lit(run_date).cast("date")
)

base_insurance_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"`{env}-cdh-delivery`.internalreports.ip_base_insurance")
spark.sql(f"OPTIMIZE `{env}-cdh-delivery`.internalreports.ip_base_insurance")
base_insurance_df = spark.table(f"`{env}-cdh-delivery`.internalreports.ip_base_insurance")

In [0]:
# # -- ============================================================
# # -- ts (Transformed Insurance Set)
# # --
# # -- This perimeter converts the base insurance perimeter into
# # -- the final business-friendly insurance structure.
# # -- It:
# # -- - Renames columns
# # -- - Applies business mappings (pool, group, property status)
# # -- - Ensures correct date casting
# # -- ============================================================

ts_df = (
    base_insurance_df
    .filter(F.col("AgreementNumber").isNotNull())
    .select(
        F.col("ThesisId"),
        F.col("MasterId"),
        F.col("AgreementNumber").alias("policy_number"),
        F.col("InsuranceStartDate").cast("date").alias("insurance_start_date"),
        F.col("InsuranceEndDate").cast("date").alias("insurance_end_date"),
        F.col("InsuranceCompany").alias("insurance_company"),
        F.when(F.col("GroupInsuranceflag"), "Group").otherwise("Solo").alias("insurance_group"),
        F.col("isactiveflag").alias("insurance_active_flag"),
        F.col("action"),
        F.col("cepal_key"),
        F.when(F.col("realestateactive") == "Active", F.lit(1)).otherwise(F.lit(0)).alias("property_status"),
        F.col("NetInsuranceAmount").alias("net_insurance_amount"),
        F.col("ForecastInsuranceAmount").alias("forecast_insurance_amount"),
        F.when(F.col("realestateactive") == "Active", F.lit(1)).otherwise(F.lit(0)).alias("property_active_flag"),
        F.col("collateralstatus").alias("collateral_status"),
        F.col("AgreementKey").alias("agreement_key"),
        F.col("CollateralKey").alias("asset_id"),
        F.col("PrimaryCommercialGR").alias("primary_commercial_name"),
        F.coalesce(F.col("TaxRegistrationNumber"), F.lit("")).alias("tax_registration_number"),
        F.col("CDI").alias("cdi"),
        F.lit("Primary Owner").alias("relation_type"),
        F.col("PortfolioName"),
        F.col("InsurancePool").alias("insurance_pool"),
        F.when(F.col("InsurancePool") == "Umbrella", F.col("InsurancePool"))
         .otherwise(
             F.when(F.col("Prefix") == "NEP", F.lit("NEPTUNE"))
              .when(F.col("Prefix") == "SYM", F.lit("SYMBOL"))
              .when(F.col("Prefix") == "LGT", F.lit("LIGHT"))
              .when(F.col("Prefix").isin("CSR", "CSW"), F.lit("COSMOS"))
              .when(F.col("Prefix").isin("HRA", "HRB", "HRD"), F.lit("HERMES"))
              .when(F.col("Prefix").isin("GLA", "GLB", "GLC", "GXR"), F.lit("GALAXY"))
              .when(F.col("Prefix") == "EUC", F.lit("EUCLID"))
              .when(F.col("Prefix").isin("MZR", "MZW", "MZD"), F.lit("MONZA"))
         ).alias("dwh_ins_pool"),
        F.col("InsuranceReceivedDate").cast("date").alias("insurance_received_date"),
        F.col("InsuranceEnteredDate").cast("date").alias("insurance_entered_date"),
        F.col("AgreementNumber").alias("policy_number_in")
    )
    .dropDuplicates()
)

ts_df = ts_df.withColumn(
    "BusinessDate", F.lit(run_date).cast("date")
)

ts_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"`{env}-cdh-delivery`.internalreports.ip_ts")
spark.sql(f"OPTIMIZE `{env}-cdh-delivery`.internalreports.ip_ts")
ts_df = spark.table(f"`{env}-cdh-delivery`.internalreports.ip_ts")
ts_df.createOrReplaceTempView("ts")

In [0]:

# -- ============================================================
# -- perimetern (Additive Insurance Perimeter)
# --
# -- This perimeter contains additive / child insurance
# -- records i.e. insurance actions that:
# -- - Do not represent standalone contracts
# -- - Refer to an existing (parent) insurance contract
# --
# -- These records are identified by having a non-null
# -- MasterId, which links them to a base insurance in ts.
# --
# -- Examples of additive actions include:
# -- - Renewals
# -- - Endorsements
# -- - Amendments or corrections
# --
# -- The purpose of this perimeter is to capture all insurance
# -- changes that must be applied on top of a base contract.
# -- ============================================================

perimetern_df = (
    base_insurance_df
    .filter(F.col("MasterId").isNotNull())
    .select(
        F.col("ThesisId"),
        F.col("MasterId"),
        F.col("AgreementNumber").alias("policy_number"),
        F.col("InsuranceStartDate").cast("date").alias("insurance_start_date"),
        F.col("InsuranceEndDate").cast("date").alias("insurance_end_date"),
        F.col("InsuranceCompany").alias("insurance_company"),
        F.when(F.col("GroupInsuranceflag"), "Group").otherwise("Solo").alias("insurance_group"), 
        F.col("isactiveflag").alias("insurance_active_flag"),
        F.col("action"),
        F.col("cepal_key"),
        F.col("Property_status") if "Property_status" in base_insurance_df.columns 
            else F.when(F.col("realestateactive") == "Active", 1)
            .otherwise(0)
            .alias("property_status"),
        F.col("NetInsuranceAmount").alias("net_insurance_amount"),
        F.col("ForecastInsuranceAmount").alias("forecast_insurance_amount"),
        F.when(F.col("realestateactive") == "Active", 1).otherwise(0).alias("property_active_flag"),
        F.col("collateralstatus").alias("collateral_status"),
        F.col("AgreementKey").alias("agreement_key"),
        F.col("PrimaryCommercialGR").alias("primary_commercial_name"),
        F.coalesce(F.col("TaxRegistrationNumber"), F.lit("")).alias("tax_registration_number"),
        F.col("CDI").alias("cdi"),
        F.lit("Primary Owner").alias("relation_type"),
        F.col("PortfolioName"),
        F.col("InsurancePool").alias("insurance_pool"),
        F.when(F.col("Prefix") == "NEP", "NEPTUNE")
         .when(F.col("Prefix") == "SYM", "SYMBOL")
         .when(F.col("Prefix") == "LGT", "LIGHT")
         .when(F.col("Prefix").isin("CSR","CSW"), "COSMOS")
         .when(F.col("Prefix").isin("HRA","HRB","HRD"), "HERMES")
         .when(F.col("Prefix").isin("GLA","GLB","GLC","GXR"), "GALAXY")
         .when(F.col("Prefix") == "EUC", "EUCLID")
         .when(F.col("Prefix").isin("MZR","MZW","MZD"), "MONZA")
         .alias("dwh_ins_pool"),
        F.col("InsuranceReceivedDate").cast("date").alias("insurance_received_date"),
        F.col("InsuranceEnteredDate").cast("date").alias("insurance_entered_date")
    )
    .dropDuplicates(["ThesisId", "MasterId", "cepal_key", "policy_number"])
)

perimetern_df = perimetern_df.withColumn(
    "BusinessDate", F.lit(run_date).cast("date")
)

perimetern_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"`{env}-cdh-delivery`.internalreports.ip_perimetern")
spark.sql(f"OPTIMIZE `{env}-cdh-delivery`.internalreports.ip_perimetern")
perimetern_df = spark.table(f"`{env}-cdh-delivery`.internalreports.ip_perimetern")

# -- ============================================================
# -- perimetern1 (Matched Additive Actions)
# -- ============================================================

perimetern1_df = (
    perimetern_df.alias("p")
    .join(
        ts_df.alias("t"),
        (F.col("p.MasterId") == F.col("t.ThesisId")) &
        (F.col("p.cepal_key") == F.col("t.cepal_key")),
        "inner"
    )
    .select(
        F.col("p.*"),
        F.col("t.policy_number_in")
    )
    .dropDuplicates()
)
perimetern1_df.createOrReplaceTempView("perimetern1")


# -- ============================================================
# -- perimetern2 (Remaining unmatched additives)
# -- ============================================================

perimetern2_df = (
    perimetern_df.alias("p")
    .join(
        perimetern1_df.select("MasterId","cepal_key").alias("p1"),
        (F.col("p.MasterId") == F.col("p1.MasterId")) &
        (F.col("p.cepal_key") == F.col("p1.cepal_key")),
        "left_anti"
    )
    .join(
        ts_df.alias("t"),
        (F.col("p.MasterId") == F.col("t.ThesisId")) &
        (F.col("p.cepal_key") == F.col("t.cepal_key")),
        "inner"
    )
    .select(
        F.col("p.*"), 
        F.col("t.policy_number_in"))
    .dropDuplicates()
)
perimetern2_df.createOrReplaceTempView("perimetern2")


# -- ============================================================
# -- perimeter1 (Full insurance universe)
# -- ============================================================

perimeter1_ts = (
    ts_df
    .select(
        "ThesisId",
        "MasterId",
        "policy_number",
        "insurance_start_date",
        "insurance_end_date",
        "insurance_company",
        "insurance_group",
        "insurance_active_flag",
        "action",
        "cepal_key",
        "property_status",
        "net_insurance_amount",
        "forecast_insurance_amount",
        "property_active_flag",
        "collateral_status",
        "agreement_key",
        "primary_commercial_name",
        "tax_registration_number",
        "cdi",
        "relation_type",
        "PortfolioName",
        "insurance_pool",
        "dwh_ins_pool",
        "insurance_received_date",
        "insurance_entered_date",
        "policy_number_in",
        "BusinessDate"
    )
)


perimeter1_df = (
    perimeter1_ts
    .unionByName(spark.table("perimetern1"))
    .unionByName(spark.table("perimetern2"))
    .dropDuplicates()
)
perimeter1_df.createOrReplaceTempView("perimeter1")


# -- ============================================================
# -- perimeter1_clean
# -- ============================================================

perimeter1_clean_df = (
    perimeter1_df
    .filter(
        ~(
            (F.col("insurance_pool").isNotNull()) &
            (F.col("insurance_group") != F.lit("Solo")) &
            (F.upper(F.col("insurance_pool")) != F.upper(F.when(F.col("dwh_ins_pool").isNull(), F.lit("")).otherwise(F.col("dwh_ins_pool"))))
        )
    )
)
perimeter1_clean_df.createOrReplaceTempView("perimeter1_clean")


# -- ============================================================
# -- perimeter_solo (ranked solo policies)
# -- ============================================================

w_solo = Window.partitionBy(F.col("agreement_key")).orderBy(F.col("insurance_start_date").desc())

perimeter_solo_df = (
    perimeter1_clean_df
    .filter(F.col("insurance_group") == F.lit("Solo"))
    .withColumn("rnk", F.row_number().over(w_solo))
    .dropDuplicates() 
)
perimeter_solo_df.createOrReplaceTempView("perimeter_solo")


# ------------------------------------------------------------
# perimeter:
#
# - Keeps a curated set of columns from perimeter1_clean
# - Applies action multiplier to Net Premium: Καθαρό Ασφάλιστρο * action
# - Left joins perimeter_solo to enrich each row with Solo insurance details
#   Join logic:
#   - same property (cepal_key)
#   - same contract (agreement_key) to avoid duplicates
#   - different insurance reference (Αρ Ασφ Συμβολαίου In) to avoid self-join
# ------------------------------------------------------------

perimeter_df = (
    perimeter1_clean_df.alias("p1")
    .join(
        perimeter_solo_df.alias("ps"),
        (F.col("p1.cepal_key") == F.col("ps.cepal_key")) &
        (F.col("p1.agreement_key") == F.col("ps.agreement_key")) &
        (F.col("p1.policy_number_in") != F.col("ps.policy_number_in")),
        how="left"
    )
    .select(
        F.col("p1.ThesisId"),
        F.col("p1.MasterId"),
        F.col("p1.policy_number"),                  
        F.col("p1.insurance_start_date"),           
        F.col("p1.insurance_end_date"),             
        F.col("p1.insurance_company"),              
        F.col("p1.insurance_group"),                
        F.col("p1.insurance_active_flag"),          
        F.col("p1.cepal_key"),
        F.col("p1.property_status"),
        (F.col("p1.net_insurance_amount") * F.col("p1.action")).alias("net_insurance_amount"),  
        F.col("p1.forecast_insurance_amount"),
        F.col("p1.property_active_flag"),
        F.col("p1.collateral_status"),
        F.col("p1.agreement_key"),                 
        F.col("p1.primary_commercial_name"),       
        F.col("p1.tax_registration_number"),       
        F.col("p1.cdi"),
        F.col("p1.relation_type"),
        F.col("p1.policy_number_in"),
        F.col("p1.PortfolioName"),
        F.col("ps.policy_number").alias("solo_policy_number"),                 
        F.col("ps.insurance_start_date").alias("solo_insurance_start_date"),   
        F.col("ps.insurance_end_date").alias("solo_insurance_end_date"),       
        F.col("ps.insurance_received_date").alias("solo_insurance_received_date"), 
        F.col("ps.insurance_entered_date").alias("solo_insurance_entered_date"),   
        F.col("ps.rnk").alias("rank_solo") ,
        F.col("p1.BusinessDate")                                     
    )
    .dropDuplicates()
)

perimeter_df.write.format("delta").mode("overwrite").saveAsTable(f"`{env}-cdh-delivery`.internalreports.ip_perimeter")
spark.sql(f"OPTIMIZE `{env}-cdh-delivery`.internalreports.ip_perimeter")
perimeter_df = spark.table(f"`{env}-cdh-delivery`.internalreports.ip_perimeter")
perimeter_df.createOrReplaceTempView("perimeter")

In [0]:
# ============================================================
# Exclusion of real estates based on Solo vs Group overlap
# ------------------------------------------------------------
# Business logic:
# Exclude real estates that satisfy ALL of the following:
# 1) Solo insurance start date < Group insurance end date
# 2) Solo insurance end date   > Group insurance start date
# 3) Solo insurance entry date > Group insurance end date
# 4) Insurance Group = 'Group'
# ============================================================

excluded_real_estate_df = (
    perimeter_df
    .filter(
        (F.col("solo_policy_number").isNotNull()) &
        (F.col("solo_insurance_start_date") < F.col("insurance_end_date")) &
        (F.col("solo_insurance_end_date") > F.col("insurance_start_date")) &
        (F.col("solo_insurance_entered_date") > F.col("insurance_end_date")) &
        (F.col("insurance_group") == "Group")
    )
    .select("cepal_key")
    .dropDuplicates()
)
excluded_real_estate_df.createOrReplaceTempView("excluded_real_estate")


# ============================================================
# Remove excluded real estates from perimeter
# and keep only the primary Solo insurance (Rank_Solo = 1)
# ============================================================

perimeter_filtered_df = (
    perimeter_df.alias("p")
    .join(
        excluded_real_estate_df.alias("ere"),
        on="cepal_key",
        how="left_anti"
    )
    .filter((F.col("p.rank_solo").isNull()) | (F.col("p.rank_solo") == F.lit(1)))
)

perimeter_filtered_df.createOrReplaceTempView("perimeter_filtered")


# ============================================================
# Normalize AFM:
# - Convert literal 'NULL' strings to actual nulls
# - Left-pad with zeros to enforce 9-digit format
# ============================================================

perimeter_final_df = (
    perimeter_filtered_df
    .withColumn(
        "tax_registration_number",
        F.when(F.col("tax_registration_number") == "NULL", None)
         .when(F.col("tax_registration_number").isNotNull(),
               F.lpad(F.col("tax_registration_number"), 9, "0"))
         .otherwise(None)
    )
)

perimeter_final_df = perimeter_final_df.select(
    "cepal_key",
    "ThesisId",
    "MasterId",
    "policy_number",                
    "insurance_start_date",         
    "insurance_end_date",           
    "insurance_company",            
    "insurance_group",              
    "insurance_active_flag",        
    "property_status",
    "net_insurance_amount",         
    "forecast_insurance_amount",    
    "property_active_flag",
    "collateral_status",
    "agreement_key",                
    "primary_commercial_name",      
    "tax_registration_number",      
    "cdi",
    "relation_type",
    "policy_number_in",             
    "PortfolioName",
    "solo_policy_number",           
    "solo_insurance_start_date",    
    "solo_insurance_end_date",      
    "solo_insurance_received_date", 
    "solo_insurance_entered_date",  
    "rank_solo",
    "BusinessDate"                     
)

perimeter_final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"`{env}-cdh-delivery`.internalreports.ip_perimeter_final")
spark.sql(f"OPTIMIZE `{env}-cdh-delivery`.internalreports.ip_perimeter_final")
perimeter_final_df = spark.table(f"`{env}-cdh-delivery`.internalreports.ip_perimeter_final")
perimeter_final_df.createOrReplaceTempView("perimeter_final")


In [0]:
# ============================================================
# Identify Slave Accounts from perimeter contracts:
# - Filter dexposure_profits to keep only Slave exposures (SlaveFlag=True)
# - Join perimeter_final_df on AgreementKey to link only relevant agreements
# - Extract the ExposureKey as LoanId
# - Remove duplicates to ensure each slave loan appears only once
# ============================================================

dexposure_filtered_slave = dexposure_profits.filter(
    F.col("SlaveFlag") == True
)

slave_accounts_df = (
    perimeter_final_df.alias("p")
    .join(
        dexposure_filtered_slave.alias("dex"),
        F.col("dex.AgreementKey") == F.col("p.agreement_key"),
        how = "left"
    )
    .select(F.col("dex.ExposureKey").alias("LoanId"))
    .dropDuplicates()
)

slave_accounts_df.createOrReplaceTempView("SlaveAccounts")


# ============================================================
# Extract Loans linked to selected Products from profits:
# - Join perimeter_final_df with dexposure_profits to limit to relevant agreements
# - Filter exposures for the predefined product keys (product_keys_df)
# - Keep only the ExposureKey as LoanId
# - Remove duplicates to ensure each loan appears only once
# ============================================================

product_keys = [
    '40113;40115','40123;40115','40587;40115','41026;40115','42600;40115','43212;40115',
    '40011;1012','40011;7800','40011;7810','41022;7363','41027;4015','41027;4016',
    '41027;6078','41027;6079','41027;7362','41027;7363','41026;4044','41026;6080',
    '41026;7365','40517;4044','40517;7365','40567;7365','40587;7365','40004;40004',
    '40011;4008','40011;4023','40011;4900','40011;4901','40011;4902','40113;40113',
    '40123;40123','40322;40325','40324;40324','40517;7362','40517;7363',
    '40567;4015','40567;4016','40567;4044','40567;6078','40567;6079','40567;6080',
    '40568;40000','40568;40001','40568;4015','40568;4016','40568;6078','40568;6079',
    '40568;7362','40568;7363','40587;4044','40587;6080','40003;4933','40011;4022',
    '40011;4025','40011;4605','40011;4910','40011;6024','40011;7343','40011;7467',
    '40011;7475','40011;7809','40011;95000','40011;95250','41021;7365','41022;7362'
]

product_keys_df = spark.createDataFrame([(k,) for k in product_keys], ["ProductKey"])

dex_small = dexposure_profits.select(
    F.col("agreementkey").alias("agreementkey"),
    F.col("SourceSystem").alias("SourceSystem"),
    F.col("ProductKey").alias("ProductKey"),
    F.col("exposureKey").alias("exposureKey"),
)

products_profits_df = (
    perimeter_final_df.alias("p")
    .join(
        dex_small.alias("dex"),
        F.trim(F.col("dex.agreementkey").cast("string")) ==
        F.trim(F.col("p.agreement_key").cast("string")),
        "left"
    )
    .join(F
          .broadcast(product_keys_df), 
          on="ProductKey", 
          how="left_semi")
    .select(F.col("dex.exposureKey").alias("LoanId"))
    .distinct()
)

products_profits_df.createOrReplaceTempView("ProductsProfits")


# ============================================================
# L1L2Subsidy:
# - Select exposures flagged as IsSubsidyFlag
# - Deduplicate to get unique LoanIds
# ============================================================

l1l2_subsidy_df = (
    dexposure
    .filter(F.col("IsSubsidyFlag") == True)
    .select(F.col("exposureKey").alias("LoanId"))
    .dropDuplicates()
)

l1l2_subsidy_df.createOrReplaceTempView("L1L2Subsidy") 


# ============================================================
# Exclusions:
# - Combine all LoanIds that should be excluded from further processing
# - Sources:
#   1) SlaveAccounts: loans linked via SlaveFlag
#   2) ProductsProfits: loans linked to selected products
#   3) L1L2Subsidy: exposures marked as IsSubsidyFlag
# - Union all sources and remove duplicates to create a single exclusions set
# ============================================================

exclusions_df = (
    slave_accounts_df
    .unionByName(products_profits_df)
    .unionByName(l1l2_subsidy_df)
    .dropDuplicates()
)

exclusions_df.createOrReplaceTempView("Exclusions")


In [0]:
# ============================================================
# OutstandingTotalBalanceAgreements_part1
# - Join perimeter with dexposure and product
# - Exclude loans present in the exclusions list
# - Filter out products containing '%ΑΙΡΕΣΗ%'
# - Keep relevant columns and drop duplicates
# ============================================================

dexposure_filtered_otb = (
    dexposure_profits
    .filter(
        F.col("OutstandingTotalBalance") != 0
        )
    .select(
            "AgreementKey",
            "ProductKey",
            "ExposureKey",
            "OutstandingTotalBalance",
            "NextInstallmentAmount",
            "NextInstallmentDate",
            "ExposureStatusCode",
            "ExposureStatus"
        )
)

product_filtered = (
    product
    .filter(~F.col("ProductDescription").like("%ΑΙΡΕΣΗ%"))
    .select("ProductKey", "ProductDescription", "SubProduct")  
)

OutstandingTotalBalanceAgreements_part1_df = (
    perimeter_final_df.alias("p")
    .join(
        dexposure_filtered_otb.alias("dex"),
        (F.col("dex.AgreementKey") == F.col("p.agreement_key"))
    )
    .join(
        product_filtered.alias("pr"),
        F.col("pr.ProductKey") == F.col("dex.ProductKey"),
        how="inner"
    )
    .join(
        exclusions_df.alias("ex"),
        F.col("dex.exposureKey") == F.col("ex.LoanId"),
        how="left_anti"  
    )
    .select(
        F.col("p.cepal_key"),
        F.col("p.agreement_key"),
        F.col("p.policy_number"),
        F.col("p.policy_number_in"),
        F.col("p.net_insurance_amount"),
        F.col("p.collateral_status"),
        F.col("dex.AgreementKey"),
        F.col("dex.ExposureKey"),
        F.col("dex.ExposureStatus"),
        F.col("dex.ExposureStatusCode"),
        F.col("dex.OutstandingTotalBalance").alias("OutstandingTotalBalanceAgr"),
        F.col("dex.NextInstallmentAmount")
    )
    .dropDuplicates()
    )


# ============================================================
# OutstandingTotalBalanceAgreements_part2
# - Compute row_number per property (cepal_key)
# - Order by ExposureStatusCode ascending, then OutstandingTotalBalance descending
# - The top loan will have rnk = 1
# ============================================================

w = Window.partitionBy("cepal_key").orderBy(
    F.col("ExposureStatusCode").asc(),
    F.col("OutstandingTotalBalanceAgr").desc()
)

OutstandingTotalBalanceAgreements_part2_df = OutstandingTotalBalanceAgreements_part1_df.withColumn(
    "rnk", F.row_number().over(w)
)


# ============================================================
# OutstandingTotalBalanceAgreements
# - Keep only the top loan per property (rnk = 1)
# ============================================================

OutstandingTotalBalanceAgreements_df = OutstandingTotalBalanceAgreements_part2_df.filter(
    F.col("rnk") == 1
)

OutstandingTotalBalanceAgreements_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"`{env}-cdh-delivery`.internalreports.ip_OutstandingTotalBalanceAgreements")

OutstandingTotalBalanceAgreements_df = spark.table(f"`{env}-cdh-delivery`.internalreports.ip_OutstandingTotalBalanceAgreements")

OutstandingTotalBalanceAgreements_df.createOrReplaceTempView("OutstandingTotalBalanceAgreements")


# ============================================================
# PaymentSchedule
# - Filter all installments after the business date
# - Find the earliest installment date per ExposureKey
# - Join back to get amounts for those dates and sum them
# ============================================================

dexposure_filtered = dexposure_profits.filter(
    (F.col("NextInstallmentDate") > F.col("BusinessDate"))
)

min_installment_df = (
    dexposure_filtered.alias("dex")
    .groupBy("ExposureKey")
    .agg(F.min("NextInstallmentDate").alias("InstallmentDate"))
)

PaymentSchedule_df = (
    dexposure_filtered.alias("dex")
    .join(
        min_installment_df.alias("m"),
        (F.col("dex.ExposureKey") == F.col("m.ExposureKey")) &
        (F.col("dex.NextInstallmentDate") == F.col("m.InstallmentDate")),
        how="inner"
    )
    .groupBy("dex.ExposureKey", "m.InstallmentDate")
    .agg(F.sum("dex.NextInstallmentAmount").alias("NextInstallment"))
    .select(
        F.col("ExposureKey"),
        F.to_date("InstallmentDate").alias("InstallmentDate"),
        F.col("NextInstallment")
    )
)

PaymentSchedule_df.createOrReplaceTempView("PaymentSchedule")


# ============================================================
# OutstandingTotalBalanceLoans
# - SIMPLIFIED: Use already-ranked OutstandingTotalBalanceAgreements
# - Just add PaymentSchedule information
# - No need to re-rank since we already have top 1 per property
# ============================================================

OutstandingTotalBalanceLoans_df = (
    OutstandingTotalBalanceAgreements_df.alias("ota")
    .join(
        PaymentSchedule_df.alias("ps"),
        F.col("ota.ExposureKey") == F.col("ps.ExposureKey"),
        how="left"
    )
    .select(
        F.col("ota.cepal_key"),
        F.col("ota.policy_number"),
        F.col("ota.policy_number_in"),
        F.col("ota.net_insurance_amount"),
        F.col("ota.collateral_status"),
        F.col("ota.ExposureKey"),
        F.col("ota.ExposureStatus"),
        F.col("ota.ExposureStatusCode"),
        F.col("ota.OutstandingTotalBalanceAgr").alias("OutstandingTotalBalanceLoan"),
        F.col("ps.NextInstallment"),
        F.col("ps.InstallmentDate").alias("NextInstallmentDate"),
        F.col("ota.rnk")
    )
)

OutstandingTotalBalanceLoans_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"`{env}-cdh-delivery`.internalreports.ip_OutstandingTotalBalanceLoans")

OutstandingTotalBalanceLoans_df = spark.table(f"`{env}-cdh-delivery`.internalreports.ip_OutstandingTotalBalanceLoans")

OutstandingTotalBalanceLoans_df.createOrReplaceTempView("OutstandingTotalBalanceLoans")


In [0]:
# ============================================================
# InLawFlag
# - Join perimeter to Party and UnderLaw to get L3869 flag
# - Filter only for UnderLawType 'N3869'
# - Convert empty strings to null for proper null handling
# - Drop duplicates to avoid multiple rows per CRS
# ============================================================

UnderLaw_filtered = (
    UnderLaw
    .filter(F.col("UnderLawTypeCode") == "N3869")
    .select("InLawFlag", 
            "CDI") 
)

inlawflag_df = (
    perimeter_final_df.alias("p")
    .join(party.alias("party"), 
          F.col("party.CDI") == F.col("p.cdi"), 
          how="inner")
    .join(UnderLaw_filtered.alias("ul"), 
          F.col("ul.CDI") == F.col("party.CDI"), 
          how="inner")
    .select(
        F.col("p.cdi"),
        F.when(
            (F.col("ul.InLawFlag") == "") | (F.col("ul.InLawFlag").isNull()),
            F.lit(None)
        ).otherwise(F.col("ul.InLawFlag")).alias("L3869_Flag")
    )
    .dropDuplicates()
)

inlawflag_df.createOrReplaceTempView("inlawflag")


# ============================================================
# TransactionFlag
# - Map CaseCode to TransactionFlag description
# - Assign SourceSystemId based on SourceSystem logic
# - Drop duplicates to avoid multiple rows per CaseCode
# ============================================================

TransactionFlag_df = (
    casecodeview.alias("cc")
    .select(
        F.col("cc.CaseCodeKey").alias("CaseCode"),
        F.col("cc.TransactionPerimeterCode").alias("TransactionFlag"),
        F.when(F.col("cc.SourceSystem") == "qc", F.lit(2))
         .otherwise(F.lit(12465))
         .alias("SourceSystemId")
    )
    .dropDuplicates()
)

TransactionFlag_df.createOrReplaceTempView("TransactionFlag")

In [0]:
# ============================================================
# Final loan selection per property (part1)
# - Matches on-prem logic:
#   1. Cell 13 pre-selected ONE loan per property (like #OutstandingTotalBalanceLoans)
#   2. Here we join dexposure on BOTH agreement_key AND ExposureKey
#   3. This ensures only the specific pre-selected loan is included
# - Filter to status codes 1-4 (Active, Overdue, Define Delay, Written Off)
# - Mark selected accounts with "Account Selected"
# ============================================================
base = (
    perimeter_final_df.alias("p")
    .join(
        OutstandingTotalBalanceLoans_df.alias("l"),
        F.col("l.cepal_key") == F.col("p.cepal_key"),
        "inner"
    )
    .join(
        dexposure_profits.alias("dex"),
        (F.col("dex.AgreementKey") == F.col("p.agreement_key")) &
        (F.col("dex.ExposureKey") == F.col("l.ExposureKey")) &  # <-- Match specific selected loan
        (F.col("dex.ExposureStatusCode").isin(["1", "2", "3", "4"])),
        "inner"
    )
    .join(
        product.alias("pr"),
        F.col("pr.ProductKey") == F.col("dex.ProductKey"),
        "left"
    )
    .join(
        dparty.alias("dparty"),
        F.col("dparty.CDI") == F.col("p.cdi"),
        "left"
    )
    .join(
        casecodeview.alias("cc"),
        F.col("cc.CaseCodeKey") == F.col("dex.CaseCode"),
        "left"
    )
    .join(
        TransactionFlag_df.alias("tf"),
        F.col("tf.CaseCode") == F.col("cc.CaseCodeKey"),
        "left"
    )
    .select(
        F.col("p.cepal_key").alias("cepal_key"),
        F.col("dex.ExposureStatusCode").alias("ExposureStatusCode"),
        F.expr("try_cast(nullif(trim(cast(dex.ExposureStatusCode as string)), '') as int)").alias("ExposureStatusCode_int"),
        F.col("p.Property_Active_flag"),
        F.col("p.collateral_status"),
        F.col("dex.ExposureKey").alias("LoanAccount"),
        F.col("p.net_insurance_amount"),
        F.col("p.forecast_insurance_amount"),
        F.col("l.OutstandingTotalBalanceLoan"),
        F.col("dex.ExposureStatus").alias("LoanStatus"),
        F.col("p.tax_registration_number").alias("AFM"),
        F.col("p.cdi"),
        F.col("p.primary_commercial_name"),
        F.col("dparty.GroupCode"),
        F.col("dparty.GroupName"),
        F.col("p.insurance_group"),
        F.col("p.insurance_company"),
        F.col("p.agreement_key"),
        F.col("p.policy_number"),
        F.col("p.policy_number_in"),
        F.col("p.insurance_start_date"),
        F.col("p.insurance_end_date"),
        F.col("p.relation_type"),
        F.col("dex.PartyName"),
        F.col("dex.ProductKey"),
        F.col("pr.ProductDescription"),
        F.col("pr.SubProduct"),
        F.col("l.NextInstallmentDate").alias("NextInstallmentDate"),
        F.col("l.NextInstallment"),
        F.col("p.PortfolioName"),
        F.col("p.solo_policy_number"),
        F.col("p.solo_insurance_start_date"),
        F.col("p.solo_insurance_end_date"),
        F.col("p.solo_insurance_received_date"),
        F.col("p.solo_insurance_entered_date"),
        F.col("cc.CaseCodeKey").alias("CaseCode"),
        F.col("tf.TransactionFlag"),
        F.col("p.BusinessDate")
    )
)

# ============================================================
# Min status logic (OPTIONAL - now redundant since we already
# filtered to the one selected loan via ExposureKey match)
# Keeping it for backward compatibility and explicit clarity
# ============================================================

min_status_df = (
    base.groupBy("cepal_key")
        .agg(F.min("ExposureStatusCode_int").alias("min_status_int"))
).alias("ms")

FinalReport_part1a = (
    base.alias("b")
    .join(
        min_status_df,
        (F.col("b.cepal_key") == F.col("ms.cepal_key")) &
        (F.col("b.ExposureStatusCode_int") == F.col("ms.min_status_int")),
        "inner"
    )
    .drop(F.col("ms.cepal_key"))
    .drop("min_status_int")
    .dropDuplicates()
)

FinalReport_part1a.createOrReplaceTempView("FinalReport_part1a")

FinalReport_part1 = (
    FinalReport_part1a
    .withColumn("Loan_Selection_Flag", F.lit("Account Selected"))
)

FinalReport_part1.createOrReplaceTempView("FinalReport_part1")


# ============================================================
# - Below there are properties (cepal_key) that do NOT have any selected loans
#   from FinalReport_part1, and are not in the exclusions list
# - LoanAccount is kept if exists, otherwise 0 / placeholder values
# - OutstandingTotalBalanceLoan = 0, LoanStatus = ''
# - Filter out products containing '%ΑΙΡΕΣΗ%'
# - Filter out exposures with OutstandingTotalBalance = 0
# - Deduplicate to one row per cepal_key using ranking
# ============================================================

rest_df_pre = (
    perimeter_final_df.alias("p")
    .join(dparty.alias("dparty"),
          F.col("dparty.CDI") == F.col("p.cdi"),
          how="left")
    .join(dexposure.alias("dex"),
          F.col("dex.AgreementKey") == F.col("p.agreement_key"),
          how="left")
    .filter(
        (F.col("dex.OutstandingTotalBalance").isNull()) |
        (F.col("dex.OutstandingTotalBalance") != 0)
    )
    .join(product_filtered.alias("pr"),
          F.col("pr.ProductKey") == F.col("dex.ProductKey"),
          how="left")
    .join(casecodeview.alias("cc"),
          F.col("cc.CaseCodeKey") == F.col("dex.CaseCode"),
          how="left")
    .join(TransactionFlag_df.alias("tf"),
          (F.col("tf.CaseCode") == F.col("cc.CaseCodeKey")) &
          (F.col("tf.SourceSystemId") == F.col("cc.SourceSystem")),
          how="left")
    .join(
        FinalReport_part1.select("cepal_key"),
        on="cepal_key",
        how="left_anti"
    )
    .join(
        exclusions_df.select(F.col("LoanId").alias("ExposureKey")).alias("ex"),
        F.col("dex.ExposureKey") == F.col("ex.ExposureKey"),
        how="left_anti"
    )
)

# Ranking: one row per cepal_key, best exposure first (same logic as cell 10)
w_rest = Window.partitionBy("p.cepal_key").orderBy(
    F.col("dex.ExposureStatusCode").asc_nulls_last(),
    F.col("dex.OutstandingTotalBalance").desc_nulls_last()
)

rest_df = (
    rest_df_pre
    .withColumn("rest_rnk", F.row_number().over(w_rest))
    .filter(F.col("rest_rnk") == 1)
    .select(
        F.col("p.cepal_key"),
        F.col("p.property_active_flag"),
        F.col("p.collateral_status"),
        F.col("dex.ExposureKey").alias("LoanAccount"),
        F.col("p.net_insurance_amount"),
        F.col("p.forecast_insurance_amount"),
        F.lit(0).alias("OutstandingTotalBalanceLoan"),
        F.lit("").alias("LoanStatus"),
        F.col("p.tax_registration_number").alias("AFM"),
        F.col("p.cdi"),
        F.col("p.primary_commercial_name"),
        F.col("dparty.GroupCode"),
        F.col("dparty.GroupName"),
        F.col("p.insurance_group"),
        F.col("p.insurance_company"),
        F.col("p.agreement_key"),
        F.col("p.policy_number"),
        F.col("p.policy_number_in"),
        F.col("p.insurance_start_date"),
        F.col("p.insurance_end_date"),
        F.col("p.relation_type"),
        F.col("dex.PartyName"),
        F.col("dex.ProductKey"),
        F.col("pr.ProductDescription"),
        F.col("pr.SubProduct"),
        F.lit("").alias("NextInstallmentDate"),
        F.lit(None).alias("NextInstallment"),
        F.col("p.PortfolioName"),
        F.col("p.solo_policy_number"),
        F.col("p.solo_insurance_start_date"),
        F.col("p.solo_insurance_end_date"),
        F.col("p.solo_insurance_received_date"),
        F.col("p.solo_insurance_entered_date"),
        F.col("cc.CaseCodeKey").alias("CaseCode"),
        F.col("tf.TransactionFlag"),
        F.lit("No account selected").alias("Loan_Selection_Flag"),
        F.col("p.BusinessDate")
    )
    .dropDuplicates()
)

rest_df.createOrReplaceTempView("Rest")


# ============================================================
# FinalReport
# - Combines selected loans from FinalReport_part1 with the remaining loans from Rest
# - Ensures no duplicate records 
# - Creates a unified TempView "FinalReport" for further processing
# ============================================================

finalreport_df = (
    FinalReport_part1.select(
        "cepal_key",
        "property_active_flag",
        "collateral_status",
        "LoanAccount",
        "net_insurance_amount",
        "forecast_insurance_amount",
        "OutstandingTotalBalanceLoan",
        "LoanStatus",
        F.col("AFM"),
        "cdi",
        "primary_commercial_name",
        "GroupCode",
        "GroupName",
        "insurance_group",
        "insurance_company",
        "agreement_key",
        "policy_number",
        "policy_number_in",
        "insurance_start_date",
        "insurance_end_date",
        "relation_type",
        "PartyName",
        "ProductKey",
        "ProductDescription",
        "SubProduct",
        "NextInstallmentDate",
        F.col("NextInstallment").cast("decimal(19,2)").alias("NextInstallment"),
        "PortfolioName",
        "solo_policy_number",
        "solo_insurance_start_date",
        "solo_insurance_end_date",
        "Loan_Selection_Flag",
        "solo_insurance_received_date",
        "solo_insurance_entered_date",
        "CaseCode",
        "TransactionFlag",
        "BusinessDate"
    )
    .union(
        rest_df.select(
            "cepal_key",
            "property_active_flag",
            "collateral_status",
            "LoanAccount",
            "net_insurance_amount",
            "forecast_insurance_amount",
            "OutstandingTotalBalanceLoan",
            "LoanStatus",
            F.col("AFM"),
            "cdi",
            "primary_commercial_name",
            "GroupCode",
            "GroupName",
            "insurance_group",
            "insurance_company",
            "agreement_key",
            "policy_number",
            "policy_number_in",
            "insurance_start_date",
            "insurance_end_date",
            "relation_type",
            "PartyName",
            "ProductKey",
            "ProductDescription",
            "SubProduct",
            "NextInstallmentDate",
            F.col("NextInstallment").cast("decimal(19,2)").alias("NextInstallment"),
            "PortfolioName",
            "solo_policy_number",
            "solo_insurance_start_date",
            "solo_insurance_end_date",
            "Loan_Selection_Flag",
            "solo_insurance_received_date",
            "solo_insurance_entered_date",
            "CaseCode",
            "TransactionFlag",
            "BusinessDate"
        )
    )
    .dropDuplicates()
)


# ============================================================
# Dedup: per (cepal_key, policy_number, policy_number_in)
# - Prefer collateral_status = 'Ενεργή' over other statuses (BRD ranking)
# - Prefer actual TransactionFlag values over null / 'N/A'
# ============================================================

w_dedup = Window.partitionBy("cepal_key", "policy_number", "policy_number_in").orderBy(
    F.when(F.col("collateral_status") == "Ενεργή", 1).otherwise(0).desc(),
    F.when(F.col("TransactionFlag").isNull() | (F.col("TransactionFlag") == "N/A"), 0).otherwise(1).desc()
)

finalreport_df = (
    finalreport_df
    .withColumn("dedup_rnk", F.row_number().over(w_dedup))
    .filter(F.col("dedup_rnk") == 1)
    .drop("dedup_rnk")
)


finalreport_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"`{env}-cdh-delivery`.internalreports.ip_FinalReport")
finalreport_df = spark.table(f"`{env}-cdh-delivery`.internalreports.ip_FinalReport")
finalreport_df.createOrReplaceTempView("FinalReport")

In [0]:
# ================================================================
# Addresses
# - Retrieves active address details per CDI from FinalReport
# - Builds full address by concatenating Street and StreetNumber
# - Excludes empty / blank / 'UNKNOWN' addresses
# - Keeps only non-communication addresses (CommunicationFlag = 0)
# - Removes duplicate address records
# ================================================================

addresses_df = (
    finalreport_df.alias("f")
    .join(dparty.alias("dparty"), 
          F.col("dparty.CDI") == F.col("f.cdi"), 
          how="left")
    .join(contactdetails.alias("cd"), 
          F.col("cd.PartyKey") == F.col("dparty.PartyKey"), 
          how="left")
    .withColumn(
        "Address",
        F.concat_ws(
            " ",
            F.coalesce(F.col("cd.Street"), F.lit("")),
            F.coalesce(F.col("cd.StreetNumber"), F.lit(""))
        )
    )
    .filter(
        (F.col("Address").isin("", " ", "UNKNOWN") == False) & 
        (F.col("cd.CommunicationFlag") == 0)
    )
    .select(
        F.col("f.cdi").alias("CDI"),
        F.col("Address"),
        F.col("cd.ContactDetailType").alias("AddressType"),
        F.col("cd.CommunicationFlag")
    )
    .dropDuplicates()
    .orderBy("CDI")
)

addresses_df.write.format("delta").mode("overwrite").saveAsTable(f"`{env}-cdh-delivery`.internalreports.ip_Addresses")
addresses_df = spark.table(f"`{env}-cdh-delivery`.internalreports.ip_Addresses")
addresses_df.createOrReplaceTempView("Addresses")

In [0]:
# ================================================================
# - Enriches FinalReport with borrower and communication address data
# - Joins active Party records per CDI
# - Retrieves CommunicationFlag = 1 address details
# - Normalizes PostalCode (removes spaces) for GeographicalArea match
# - Resolves County using fallback to GeographicalArea when needed
# ================================================================

export_1_df = (
    finalreport_df.alias("f")
    .join(dparty.alias("dparty"), F.col("dparty.CDI") == F.col("f.cdi"), how="left")
    .join(
        contactdetails.alias("cd"),
        (F.col("cd.PartyKey") == F.col("dparty.PartyKey")) & (F.col("cd.CommunicationFlag") == 1),
        how="left"
    )
    .join(
        geographicalarea.alias("geo"),
        F.expr("replace(geo.PostalCode,' ','') = replace(cd.PostalCode,' ','')"),
        how="left"
    )
    .withColumn(
        "Address",
        F.concat_ws(" ", F.coalesce(F.col("cd.Street"), F.lit("")), F.coalesce(F.col("cd.StreetNumber"), F.lit("")))
    )
    .withColumn(
        "County_final",
        F.when(
            F.col("cd.County").isin("N/A", "ΝΑ ΣΥΜΠΛΗΡΩΘΕΙ"),
            F.col("geo.County")
        ).otherwise(F.col("cd.County"))
    )
    .select(
        F.col("f.cepal_key"),
        F.col("f.Property_Active_flag"),
        F.col("f.collateral_status"),
        F.col("f.LoanAccount"),
        F.col("f.net_insurance_amount"),
        F.col("f.forecast_insurance_amount"),
        F.col("f.OutstandingTotalBalanceLoan"),
        F.col("f.LoanStatus"),
        F.col("f.AFM"),
        F.col("f.cdi"),
        F.col("f.PartyName").alias("BorrowerName"),
        F.col("f.relation_type"),
        F.col("f.GroupCode"),
        F.col("f.GroupName"),
        F.col("f.insurance_group"),
        F.col("f.insurance_company"),
        F.col("f.agreement_key"),
        F.col("f.ProductKey").alias("ProductCode_profits"),
        F.col("f.ProductDescription").alias("ProductDescription_profits"),
        F.col("f.SubProduct").alias("SubProductDescription_profits"),
        F.col("f.policy_number"),
        F.col("f.policy_number_in"),
        F.col("f.insurance_start_date").cast("date"),
        F.col("f.insurance_end_date").cast("date"),
        F.col("f.solo_policy_number"),
        F.col("f.solo_insurance_start_date").cast("date"),
        F.col("f.solo_insurance_end_date").cast("date"),
        F.col("f.NextInstallmentDate"),
        F.col("f.NextInstallment"),
        F.col("f.PortfolioName"),
        F.col("Address"),
        F.col("geo.PostalCode").alias("PostalCode"),
        F.col("County_final").alias("County"),
        F.col("f.Loan_Selection_Flag"),
        F.col("f.solo_insurance_received_date").cast("date"),
        F.col("f.solo_insurance_entered_date").cast("date"),
        F.col("f.CaseCode"),
        F.coalesce(F.col("f.TransactionFlag"), F.lit("N/A")).alias("TransactionFlag"),
        F.col("f.BusinessDate")
    )
    .dropDuplicates()
    .orderBy(F.col("f.cepal_key"), F.col("f.net_insurance_amount"))
)

export_1_df.createOrReplaceTempView("export_1")


# ================================================================
# - Combines export_1 with supplemental address and inlaw flag data
# - Uses borrower CDI to join fallback addresses where the main address is empty/UNKNOWN
# - Adds L3869_Flag from inlawflag table for regulatory/flagging purposes
# - Preserves all key loan, insurance, and property fields
# - Removes duplicates to ensure one row per loan/property combination
# ================================================================

e = spark.table("export_1").alias("e")
inlawflag_df = spark.table("inlawflag").alias("inlawflag")


addresses = (
    addresses_df
    .select(
        F.col("CDI"),
        F.col("Address")
    )
    .dropDuplicates()
    .alias("Addresses")
)


final_export_df = (
    e
    .join(
        addresses,
        F.col("Addresses.CDI") == F.col("e.cdi"),
        "left"
    )
    .join(
        inlawflag_df,
        F.col("inlawflag.cdi") == F.col("e.cdi"),
        "left"
    )
    .withColumn(
        "Address_final",
        F.when(
            F.col("e.Address").isin("", " ", "UNKNOWN"),
            F.col("Addresses.Address")
        ).otherwise(F.col("e.Address"))
    )
    .select(
        F.col("e.cepal_key"),
        F.col("e.Property_Active_flag"),
        F.col("e.collateral_status"),
        F.col("e.LoanAccount"),
        F.col("e.net_insurance_amount"),
        F.col("e.forecast_insurance_amount"),
        F.col("e.OutstandingTotalBalanceLoan"),
        F.col("e.LoanStatus"),
        F.col("e.AFM"),
        F.col("e.cdi"),
        F.col("e.BorrowerName"),
        F.col("e.relation_type"),
        F.col("e.GroupCode"),
        F.col("e.GroupName"),
        F.col("e.insurance_group"),
        F.col("e.insurance_company"),
        F.col("e.agreement_key"),
        F.col("e.ProductCode_profits"),
        F.col("e.ProductDescription_profits"),
        F.col("e.SubProductDescription_profits"),
        F.col("e.policy_number"),
        F.col("e.policy_number_in"),
        F.col("e.insurance_start_date"),
        F.col("e.insurance_end_date"),
        F.col("e.solo_policy_number"),
        F.col("e.solo_insurance_start_date"),
        F.col("e.solo_insurance_end_date"),
        F.col("e.NextInstallmentDate"),
        F.col("e.NextInstallment"),
        F.col("e.PortfolioName"),
        F.col("Address_final").alias("Address"),
        F.col("e.PostalCode"),
        F.col("e.County"),
        F.col("e.Loan_Selection_Flag"),
        F.col("e.solo_insurance_received_date"),
        F.col("e.solo_insurance_entered_date"),
        F.col("e.CaseCode"),
        F.coalesce(F.col("e.TransactionFlag"), F.lit("N/A")).alias("TransactionFlag"),
        F.col("inlawflag.L3869_Flag").alias("L3869_Flag"),
        F.col("e.BusinessDate")
    )
    .dropDuplicates()
)

final_export_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"`{env}-cdh-delivery`.internalreports.ip_FinalExport")
final_export_df = spark.table(f"`{env}-cdh-delivery`.internalreports.ip_FinalExport")
final_export_df.createOrReplaceTempView("final_export")


In [0]:
# ================================================================
# - Aggregates customer-level information from final_export
# - For Customer: combines all GroupCodes, GroupNames, CDIs, addresses, postal codes, and counties per customer
# - For Customer1: combines all AFMs, CDIs, BorrowerNames, and L3869 flags per customer-agreement
# - Ensures distinct records and concatenates multiple values with ' | '
# ================================================================

customer_df = (
    final_export_df
    .select(
        "cepal_key",
        "GroupCode",
        "GroupName",
        "AFM",
        "cdi",
        "BorrowerName",
        "Address",
        "PostalCode",
        "County"
    )
    .dropDuplicates()
    .groupBy("cepal_key")
    .agg(
        F.concat_ws(" | ", F.collect_set("GroupCode")).alias("GroupCode"),
        F.concat_ws(" | ", F.collect_set("GroupName")).alias("GroupName"),
        F.concat_ws(" | ", F.collect_set("cdi")).alias("cdi"),
        F.concat_ws(" | ", F.collect_set("Address")).alias("Address"),
        F.concat_ws(" | ", F.collect_set("PostalCode")).alias("PostalCode"),
        F.concat_ws(" | ", F.collect_set("County")).alias("County")
    )
)

customer_df.createOrReplaceTempView("Customer")


customer1_df = (
    final_export_df
    .select(
        "cepal_key",
        "AFM",
        "cdi",
        "BorrowerName",
        "L3869_Flag",
        "agreement_key"
    )
    .dropDuplicates()
    .groupBy("cepal_key", "agreement_key")
    .agg(
        F.concat_ws(" | ", F.collect_set("AFM")).alias("AFM"),
        F.concat_ws(" | ", F.collect_set("cdi")).alias("cdi"),
        F.concat_ws(" | ", F.collect_set(F.col("BorrowerName").cast("string"))).alias("BorrowerName"),
        F.concat_ws(" | ", F.collect_set("L3869_Flag")).alias("L3869_Flag")
    )
)

customer1_df.createOrReplaceTempView("Customer1")

In [0]:
# ================================================================
# - Aggregate all CaseCode and TransactionFlag values per customer (cepal_key),
#   agreement (agreement_key) and loan account.
# - Concatenate multiple values with '|' so downstream reports see all codes.
# - Ensure distinct input rows before aggregation and produce deterministic ordering.
# ================================================================

casecode_df = (
    final_export_df
    .select(
        "cepal_key",
        "agreement_key",      
        "LoanAccount",        
        "CaseCode",
        "TransactionFlag"
    )
    .dropDuplicates()
    .groupBy(
        "cepal_key",
        "agreement_key",
        "LoanAccount"
    )
    .agg(
        F.concat_ws('|', F.collect_set(F.col("CaseCode").cast("string"))).alias("CaseCode"),
        F.concat_ws('|', F.collect_set(F.col("TransactionFlag").cast("string"))).alias("TransactionFlag")
    )
)

casecode_df.createOrReplaceTempView("casecode")

In [0]:
# =====================================================================================
# Orion portfolio–specific enrichment logic for Final Export
#
# Objective:
# 1) Identify Orion accounts with:
#       - GBV > 0
#       - DenouncedDate IS NULL
#       - At least 12 future ACTIVE installments
#    and classify them as "Non-Denounced". All others are classified as "Denounced".
#
# 2) Calculate CustomerPayerSegment per (LoanAccount, CDI) combination
#    using analytics metrics and portfolio hierarchy (GALAXY parent portfolio).
#
# The output consists of:
#   - Account-level Denounced / Non-Denounced flag
#   - CustomerPayerSegment per account-CDI pair
#
# All logic strictly applies to:
#   PortfolioName = 'Orion X Securitisation DAC'
# =====================================================================================

final_export_df_filtered_orion = (
    final_export_df
    .filter(F.col("PortfolioName") == "Orion X Securitisation DAC")
    )

dexposure_filtered = (
    dexposure_profits
        .filter(
            (F.col("GBV") > 0) &
            (F.col("DenouncedDate").isNull())
        )
        .select("ExposureKey")
)


# =====================================================================================
# Retrieve all Orion Loan Accounts
# =====================================================================================

Orion_Accounts_df = (
    final_export_df_filtered_orion
    .select("LoanAccount") 
    .dropDuplicates()
)

Orion_Accounts_df.createOrReplaceTempView("Orion_Accounts")


# =====================================================================================
# Keep Orion accounts satisfying GBV & DenouncedDate conditions
# =====================================================================================

Orion_Accounts_GBVpositive_DenounceDateNULL_df = (
    Orion_Accounts_df.alias("oa")
    .join(
        dexposure_filtered.alias("dex"),
        F.col("oa.LoanAccount") == F.col("dex.ExposureKey"),
        how="inner"
    )
    .select("oa.LoanAccount")
    .dropDuplicates()
)

Orion_Accounts_GBVpositive_DenounceDateNULL_df.createOrReplaceTempView("Orion_Accounts_GBVpositive_DenounceDateNULL")


# =====================================================================================
# Retrieve future installments (InstallmentDate > BusinessDate, InstallmentAmount ≠ 0)
# Join with dexposure, that only has active installments, instead of dpaymentschedule
# =====================================================================================

Orion_Accounts_GBVpositive_DenounceDateNULL_AllActiveInstallements_df = (
    Orion_Accounts_GBVpositive_DenounceDateNULL_df.alias("oagd")
    .join(dexposure_profits.alias("dex"), 
          F.col("oagd.LoanAccount") == F.col("dex.ExposureKey"),
          how="inner")
    .filter((F.col("dex.NextInstallmentDate") > F.col("dex.BusinessDate")) &
            (F.col("dex.NextInstallmentAmount") != 0)
            )
    .select("oagd.LoanAccount", "dex.NextInstallmentDate")
    .dropDuplicates()
)

Orion_Accounts_GBVpositive_DenounceDateNULL_AllActiveInstallements_df.createOrReplaceTempView("Orion_Accounts_GBVpositive_DenounceDateNULL_AllActiveInstallements")


# =====================================================================================
# Identify accounts with ≥ 12 active future installments
# =====================================================================================

Orion_NonDenounced_Accounts_df = (
    Orion_Accounts_GBVpositive_DenounceDateNULL_AllActiveInstallements_df
    .groupBy("LoanAccount")
    .agg(F.count("*").alias("InstallmentCount"))
    .filter(F.col("InstallmentCount") >= 12)
    .select("LoanAccount")
    .dropDuplicates()
)

Orion_NonDenounced_Accounts_df.createOrReplaceTempView("Orion_NonDenounced_Accounts")


# =====================================================================================
# Denounced / Non-Denounced classification
# =====================================================================================

bauexlusion_retail_Orion_Final_df = (
    final_export_df_filtered_orion.alias("fe")
    .join(
        Orion_NonDenounced_Accounts_df.alias("onna"),
        F.col("fe.LoanAccount") == F.col("onna.LoanAccount"),
        how="left"
    )
    .select(
        "fe.LoanAccount",
        F.when(
            F.col("onna.LoanAccount").isNotNull(), "Non-Denounced")
            .otherwise("Denounced")
        .alias("Account_is_Denounced_NonDenounced")
    )
    .dropDuplicates()
)

bauexlusion_retail_Orion_Final_df.createOrReplaceTempView("bauexlusion_retail_Orion_Final")


# =====================================================================================
# Prepare (LoanAccount, CDI) combinations for Orion portfolio
# =====================================================================================

Orion_Account_CDIs_df = (
    final_export_df_filtered_orion
    .select("LoanAccount", "cdi")
    .dropDuplicates()
)

Orion_Account_CDIs_df.createOrReplaceTempView("Orion_Account_CDIs")


# =====================================================================================
# Calculate CustomerPayerSegment:
#   Derived from danalyticsmetrics.PayerFlagEntity
#   Joined via: ExposureKey, PrimaryPartyCDI
#   Limited to ParentPortfolio = 'GALAXY'
# =====================================================================================

portfolio_filtered = (
    portfolio
    .filter(F.col("ParentPortfolio").isin("GALAXY"))
    )

CustomerPayerSegment_Orion_df = (
    Orion_Account_CDIs_df.alias("oac")
    .join(dexposure.alias("dex"), 
          F.col("oac.LoanAccount") == F.col("dex.ExposureKey"), 
          how="inner"
          )
    .join(danalyticsmetrics.alias("gs"), 
          (F.col("gs.ExposureKey") == F.col("dex.ExposureKey")) & 
          (F.col("gs.PrimaryPartyCDI") == F.col("dex.cdi")) & 
          (F.col("gs.PrimaryPartyCDI") == F.col("oac.cdi")),
          how="inner"
          )
    .join(portfolio_filtered.alias("p"), 
          F.col("dex.PortfolioKey") == F.col("p.unit_code_profits"), 
          how="inner"
          )
    .select(
        "oac.LoanAccount",
        "oac.cdi",
        F.substring(F.col("gs.PayerFlagEntity").cast("string"), 1, 9).alias("CustomerPayerSegment")
        )
    .dropDuplicates()
)

CustomerPayerSegment_Orion_df.createOrReplaceTempView("CustomerPayerSegment_Orion")


In [0]:
# =====================================================================================
# Galaxy II portfolio–specific enrichment logic for Final Export.
#
# Objective:
# 1) Identify Galaxy II accounts with:
#       - GBV > 0
#       - DenouncedDate IS NULL
#       - At least 12 future ACTIVE installments
#    Classify them as "Non-Denounced", others as "Denounced".
#
# 2) Calculate CustomerPayerSegment per (LoanAccount, CDI) combination
#    using analytics metrics and portfolio hierarchy (ParentPortfolio = 'GALAXY').
#
# Output:
#   - Account-level Denounced / Non-Denounced flag
#   - CustomerPayerSegment per account-CDI pair
#
# All logic strictly applies to:
#   PortfolioName = 'Galaxy II Funding DAC'
# =====================================================================================

final_export_df_filtered_galaxy = (
    final_export_df
    .filter(F.col("PortfolioName") == "Galaxy II Funding DAC")
    )

dexposure_filtered = (
    dexposure_profits
        .filter(
            (F.col("GBV") > 0) &
            (F.col("DenouncedDate").isNull())
        )
        .select("ExposureKey")
)


# =====================================================================================
# Retrieve all Galaxy II Loan Accounts
# =====================================================================================
GalaxyII_Accounts_df = (
    final_export_df_filtered_galaxy
    .select("LoanAccount")
    .dropDuplicates()
)

GalaxyII_Accounts_df.createOrReplaceTempView("GalaxyII_Accounts")


# =====================================================================================
# Keep accounts satisfying GBV & DenouncedDate conditions
# =====================================================================================
GalaxyII_Accounts_GBVpositive_DenounceDateNULL_df = (
    GalaxyII_Accounts_df.alias("g2a")
    .join(dexposure_filtered.alias("dex"), 
          F.col("g2a.LoanAccount") == F.col("dex.ExposureKey"), 
          how="inner")
    .select("g2a.LoanAccount")
    .dropDuplicates()
)

GalaxyII_Accounts_GBVpositive_DenounceDateNULL_df.createOrReplaceTempView("GalaxyII_Accounts_GBVpositive_DenounceDateNULL")


# =====================================================================================
# Retrieve future installments (InstallmentDate > BusinessDate, InstallmentAmount ≠ 0)
# Join with dexposure, that only has active installments, instead of dpaymentschedule
# =====================================================================================

GalaxyII_Accounts_GBVpositive_DenounceDateNULL_AllActiveInstallements_df = (
    GalaxyII_Accounts_GBVpositive_DenounceDateNULL_df.alias("oagd")
    .join(dexposure_profits.alias("dex"), 
          F.col("oagd.LoanAccount") == F.col("dex.ExposureKey"), 
          how="inner")
    .filter((F.col("dex.NextInstallmentDate") > F.col("dex.BusinessDate")) &
            (F.col("dex.NextInstallmentAmount") != 0)
            )
    .select("oagd.LoanAccount", "dex.NextInstallmentDate")
    .dropDuplicates()
)

GalaxyII_Accounts_GBVpositive_DenounceDateNULL_AllActiveInstallements_df.createOrReplaceTempView("GalaxyII_Accounts_GBVpositive_DenounceDateNULL_AllActiveInstallements")


# =====================================================================================
# Identify accounts with ≥ 12 active future installments
# =====================================================================================

GalaxyII_NonDenounced_Accounts_df = (
    GalaxyII_Accounts_GBVpositive_DenounceDateNULL_AllActiveInstallements_df
    .groupBy("LoanAccount")
    .agg(F.count("*").alias("InstallmentCount"))
    .filter(F.col("InstallmentCount") >= 12)
    .select("LoanAccount")
    .dropDuplicates()
)

GalaxyII_NonDenounced_Accounts_df.createOrReplaceTempView("GalaxyII_NonDenounced_Accounts")


# =====================================================================================
# Denounced / Non-Denounced classification
# =====================================================================================

bauexlusion_retail_GalaxyII_Final_df = (
    final_export_df_filtered_galaxy.alias("fe")
    .join(
        GalaxyII_NonDenounced_Accounts_df.alias("gna"),
        F.col("fe.LoanAccount") == F.col("gna.LoanAccount"),
        how="left"
    )
    .select(
        "fe.LoanAccount",
        F.when(F.col("gna.LoanAccount").isNotNull(), "Non-Denounced")
         .otherwise("Denounced")
         .alias("Account_is_Denounced_NonDenounced")
    )
    .dropDuplicates()
)

bauexlusion_retail_GalaxyII_Final_df.createOrReplaceTempView("bauexlusion_retail_GalaxyII_Final")


# =====================================================================================
# Prepare (LoanAccount, CDI) combinations for Galaxy II
# =====================================================================================

GalaxyII_Account_CDIs_df = (
    final_export_df_filtered_galaxy
    .select("LoanAccount", "cdi")
    .dropDuplicates()
)

GalaxyII_Account_CDIs_df.createOrReplaceTempView("GalaxyII_Account_CDIs")


# =====================================================================================
# Calculate CustomerPayerSegment:
#   Derived from danalyticsmetrics.PayerFlagEntity
#   Joined via: ExposureKey, PrimaryPartyCDI
#   Limited to ParentPortfolio = 'GALAXY'
# =====================================================================================

portfolio_filtered = (
    portfolio
    .filter(F.col("ParentPortfolio").isin("GALAXY"))
    )

CustomerPayerSegment_GalaxyII_df = (
    GalaxyII_Account_CDIs_df.alias("oac")
    .join(dexposure.alias("dex"), 
          F.col("oac.LoanAccount") == F.col("dex.ExposureKey"), 
          how="inner"
        )
    .join(danalyticsmetrics.alias("gs"),
          (F.col("gs.ExposureKey") == F.col("dex.ExposureKey")) &
          (F.col("gs.PrimaryPartyCDI") == F.col("dex.cdi")) &
          (F.col("gs.PrimaryPartyCDI") == F.col("oac.cdi")),
          how="inner"
        )
    .join(portfolio_filtered.alias("p"), 
          F.col("dex.PortfolioKey") == F.col("p.unit_code_profits"), 
          how="inner"
        )
    .select(
        "oac.LoanAccount",
        "oac.cdi",
        F.substring(F.col("gs.PayerFlagEntity").cast("string"), 1, 9).alias("CustomerPayerSegment")
        )
    .dropDuplicates()
)

CustomerPayerSegment_GalaxyII_df.createOrReplaceTempView("CustomerPayerSegment_GalaxyII")

In [0]:
# =====================================================================================
# Cosmos Retail portfolio–specific enrichment logic for Final Export.
#
# Objective:
# 1) Identify Cosmos Retail accounts with:
#       - GBV > 0
#       - DenouncedDate IS NULL
#       - At least 12 future ACTIVE installments
#    Classify them as "Non-Denounced", others as "Denounced".
#
# 2) Calculate CustomerPayerSegment per (LoanAccount, CDI) combination
#    using analytics metrics and portfolio hierarchy (ParentPortfolio = 'COSMOS').
#
# Output:
#   - Account-level Denounced / Non-Denounced flag
#   - CustomerPayerSegment per account-CDI pair
#
# All logic strictly applies to:
#   PortfolioName = 'Cosmos Retail'
# =====================================================================================

final_export_df_filtered_cosmos = (
    final_export_df
    .filter(F.col("PortfolioName") == "Cosmos Retail")
    )

dexposure_filtered = (
    dexposure_profits
        .filter(
            (F.col("GBV") > 0) &
            (F.col("DenouncedDate").isNull())
        )
        .select("ExposureKey")
)


# =====================================================================================
# Retrieve all Cosmos Retail Loan Accounts
# =====================================================================================

CosmosRetail_Accounts_df = (
    final_export_df_filtered_cosmos
    .select("LoanAccount")
    .dropDuplicates()
)

CosmosRetail_Accounts_df.createOrReplaceTempView("CosmosRetail_Accounts")


# =====================================================================================
# Keep accounts satisfying GBV & DenouncedDate conditions
# =====================================================================================

CosmosRetail_Accounts_GBVpositive_DenounceDateNULL_df = (
    CosmosRetail_Accounts_df.alias("cr")
    .join(dexposure_filtered.alias("dex"), 
          F.col("cr.LoanAccount") == F.col("dex.ExposureKey"), 
          how="inner")
    .select("cr.LoanAccount")
    .dropDuplicates()
)

CosmosRetail_Accounts_GBVpositive_DenounceDateNULL_df.createOrReplaceTempView("CosmosRetail_Accounts_GBVpositive_DenounceDateNULL")


# =====================================================================================
# Retrieve future installments (InstallmentDate > BusinessDate, InstallmentAmount ≠ 0)
# Join with dexposure, that only has active installments, instead of dpaymentschedule
# =====================================================================================

CosmosRetail_Accounts_GBVpositive_DenounceDateNULL_AllActiveInstallements_df = (
    CosmosRetail_Accounts_GBVpositive_DenounceDateNULL_df.alias("oagd")
    .join(dexposure_profits.alias("dex"), 
          F.col("oagd.LoanAccount") == F.col("dex.ExposureKey"), 
          how="inner")
    .filter((F.col("dex.NextInstallmentDate") > F.col("dex.BusinessDate")) &
            (F.col("dex.NextInstallmentAmount") != 0)
            )
    .select("oagd.LoanAccount", "dex.NextInstallmentDate")
    .dropDuplicates()
)

CosmosRetail_Accounts_GBVpositive_DenounceDateNULL_AllActiveInstallements_df.createOrReplaceTempView("CosmosRetail_Accounts_GBVpositive_DenounceDateNULL_AllActiveInstallements")


# =====================================================================================
# Keep only ACTIVE installments
# =====================================================================================

CosmosRetail_NonDenounced_Accounts_df = (
    CosmosRetail_Accounts_GBVpositive_DenounceDateNULL_AllActiveInstallements_df
    .groupBy("LoanAccount")
    .agg(F.count("*").alias("InstallmentCount"))
    .filter(F.col("InstallmentCount") >= 12)
    .select("LoanAccount")
    .dropDuplicates()
)

CosmosRetail_NonDenounced_Accounts_df.createOrReplaceTempView("CosmosRetail_NonDenounced_Accounts")


# =====================================================================================
# Denounced / Non-Denounced classification
# =====================================================================================

bauexlusion_retail_CosmosRetail_Final_df = (
    final_export_df_filtered_cosmos.alias("fe")
    .join(
        CosmosRetail_NonDenounced_Accounts_df.alias("gna"),
        F.col("fe.LoanAccount") == F.col("gna.LoanAccount"),
        how="left"
        )
    .select(
        "fe.LoanAccount",
        F.when(F.col("gna.LoanAccount").isNotNull(), "Non-Denounced")
         .otherwise("Denounced")
         .alias("Account_is_Denounced_NonDenounced")
        )
    .dropDuplicates()
)

bauexlusion_retail_CosmosRetail_Final_df.createOrReplaceTempView("bauexlusion_retail_CosmosRetail_Final")


# =====================================================================================
# Prepare (LoanAccount, CDI) combinations for Cosmos Retail
# =====================================================================================

CosmosRetail_Account_CDIs_df = (
    final_export_df_filtered_cosmos
    .select("LoanAccount", "cdi")
    .dropDuplicates()
)

CosmosRetail_Account_CDIs_df.createOrReplaceTempView("CosmosRetail_Account_CDIs")


# =====================================================================================
# Calculate CustomerPayerSegment:
#   Derived from danalyticsmetrics.PayerFlagEntity
#   Joined via: ExposureKey, PrimaryPartyCDI
#   Limited to ParentPortfolio = 'COSMOS'
# =====================================================================================

portfolio_filtered = (
    portfolio
    .filter(F.col("ParentPortfolio") == "COSMOS")
    )

CustomerPayerSegment_CosmosRetail_df = (
    CosmosRetail_Account_CDIs_df.alias("oac")
    .join(dexposure.alias("dex"), 
          F.col("oac.LoanAccount") == F.col("dex.ExposureKey"), 
          how="inner"
          )
    .join(danalyticsmetrics.alias("gs"),
          (F.col("gs.ExposureKey") == F.col("dex.ExposureKey")) &
          (F.col("gs.PrimaryPartyCDI") == F.col("dex.cdi")) &
          (F.col("gs.PrimaryPartyCDI") == F.col("oac.cdi")),
          how="inner")
    .join(portfolio_filtered.alias("p"), 
          F.col("dex.PortfolioKey") == F.col("p.unit_code_profits"), 
          how="inner"
          )
    .select(
        "oac.LoanAccount",
        "oac.cdi",
        F.substring(F.col("gs.PayerFlagEntity").cast("string"), 1, 9).alias("CustomerPayerSegment")
        )
    .dropDuplicates()
)

CustomerPayerSegment_CosmosRetail_df.createOrReplaceTempView("CustomerPayerSegment_CosmosRetail")

In [0]:
# =====================================================================================
# MONZA PORTFOLIO - RETAIL portfolio–specific enrichment logic for Final Export.
# All logic strictly applies to: PortfolioName = 'MONZA PORTFOLIO - RETAIL'
# =====================================================================================

final_export_df_filtered_monzaretail = (
    final_export_df
    .filter(F.col("PortfolioName") == "MONZA PORTFOLIO - RETAIL")
)

dexposure_filtered = (
    dexposure_profits
        .filter((F.col("GBV") > 0) & (F.col("DenouncedDate").isNull()))
        .select("ExposureKey")
)

MonzaRetail_Accounts_df = (
    final_export_df_filtered_monzaretail
    .select("LoanAccount")
    .dropDuplicates()
)
MonzaRetail_Accounts_df.createOrReplaceTempView("MonzaRetail_Accounts")

MonzaRetail_Accounts_GBVpositive_DenounceDateNULL_df = (
    MonzaRetail_Accounts_df.alias("mr")
    .join(dexposure_filtered.alias("dex"),
          F.col("mr.LoanAccount") == F.col("dex.ExposureKey"), how="inner")
    .select("mr.LoanAccount")
    .dropDuplicates()
)
MonzaRetail_Accounts_GBVpositive_DenounceDateNULL_df.createOrReplaceTempView("MonzaRetail_Accounts_GBVpositive_DenounceDateNULL")

MonzaRetail_Accounts_AllActiveInstallements_df = (
    MonzaRetail_Accounts_GBVpositive_DenounceDateNULL_df.alias("oagd")
    .join(dexposure_profits.alias("dex"),
          F.col("oagd.LoanAccount") == F.col("dex.ExposureKey"), how="inner")
    .filter((F.col("dex.NextInstallmentDate") > F.col("dex.BusinessDate")) &
            (F.col("dex.NextInstallmentAmount") != 0))
    .select("oagd.LoanAccount", "dex.NextInstallmentDate")
    .dropDuplicates()
)
MonzaRetail_Accounts_AllActiveInstallements_df.createOrReplaceTempView("MonzaRetail_Accounts_AllActiveInstallements")

MonzaRetail_NonDenounced_Accounts_df = (
    MonzaRetail_Accounts_AllActiveInstallements_df
    .groupBy("LoanAccount")
    .agg(F.count("*").alias("InstallmentCount"))
    .filter(F.col("InstallmentCount") >= 12)
    .select("LoanAccount")
    .dropDuplicates()
)
MonzaRetail_NonDenounced_Accounts_df.createOrReplaceTempView("MonzaRetail_NonDenounced_Accounts")

bauexlusion_retail_MONZARETAIL_Final_df = (
    final_export_df_filtered_monzaretail.alias("fe")
    .join(MonzaRetail_NonDenounced_Accounts_df.alias("gna"),
          F.col("fe.LoanAccount") == F.col("gna.LoanAccount"), how="left")
    .select(
        "fe.LoanAccount",
        F.when(F.col("gna.LoanAccount").isNotNull(), "Non-Denounced")
         .otherwise("Denounced")
         .alias("Account_is_Denounced_NonDenounced")
    )
    .dropDuplicates()
)
bauexlusion_retail_MONZARETAIL_Final_df.createOrReplaceTempView("bauexlusion_retail_MONZARETAIL_Final")

In [0]:
# =====================================================================================
# GXL One Holding portfolio–specific enrichment logic for Final Export.
#
# Objective:
# 1) Identify GXL One Holding accounts with:
#       - GBV > 0
#       - DenouncedDate IS NULL
#       - At least 12 future ACTIVE installments
#    Classify them as "Non-Denounced", others as "Denounced".
#
# 2) Prepare for CustomerPayerSegment (if needed, similar to Cosmos/Galaxy II logic)
#
# Output:
#   - Account-level Denounced / Non-Denounced flag
#
# All logic strictly applies to:
#   PortfolioName = 'GXL One Holding'
# =====================================================================================

final_export_df_filtered_glx = (
    final_export_df
    .filter(F.col("PortfolioName") == "GXL One Holding") #τσεκ
)

dexposure_filtered = (
    dexposure_profits
        .filter(
            (F.col("GBV") > 0) &
            (F.col("DenouncedDate").isNull())
        )
        .select("ExposureKey")
)


# =====================================================================================
# Retrieve all GXL One Holding Loan Accounts
# =====================================================================================

GXLOneHolding_Accounts_df = (
    final_export_df_filtered_glx
    .select("LoanAccount")
    .dropDuplicates()
)

GXLOneHolding_Accounts_df.createOrReplaceTempView("GXLOneHolding_Accounts")


# =====================================================================================
# Keep accounts satisfying GBV & DenouncedDate conditions
# =====================================================================================

GXLOneHolding_Accounts_GBVpositive_DenounceDateNULL_df = (
    GXLOneHolding_Accounts_df.alias("gh")
    .join(dexposure_filtered.alias("dex"), 
          F.col("gh.LoanAccount") == F.col("dex.ExposureKey"), 
          how="inner")
    .select("gh.LoanAccount")
    .dropDuplicates()
)

GXLOneHolding_Accounts_GBVpositive_DenounceDateNULL_df.createOrReplaceTempView("GXLOneHolding_Accounts_GBVpositive_DenounceDateNULL")


# =====================================================================================
# Retrieve future installments (InstallmentDate > BusinessDate, InstallmentAmount ≠ 0)
# Join with dexposure, that only has active installments, instead of dpaymentschedule
# =====================================================================================

GXLOneHolding_Accounts_GBVpositive_DenounceDateNULL_AllActiveInstallements_df = (
    GXLOneHolding_Accounts_GBVpositive_DenounceDateNULL_df.alias("oagd")
    .join(dexposure_profits.alias("dex"), 
          F.col("oagd.LoanAccount") == F.col("dex.ExposureKey"), 
          how="inner")
    .filter((F.col("dex.NextInstallmentDate") > F.col("dex.BusinessDate")) &
            (F.col("dex.NextInstallmentAmount") != 0)
            )
    .select("oagd.LoanAccount", "dex.NextInstallmentDate")
    .dropDuplicates()
)

GXLOneHolding_Accounts_GBVpositive_DenounceDateNULL_AllActiveInstallements_df.createOrReplaceTempView("GXLOneHolding_Accounts_GBVpositive_DenounceDateNULL_AllActiveInstallements")


# =====================================================================================
# Identify accounts with ≥ 12 active future installments
# =====================================================================================

GXLOneHolding_NonDenounced_Accounts_df = (
    GXLOneHolding_Accounts_GBVpositive_DenounceDateNULL_AllActiveInstallements_df
    .groupBy("LoanAccount")
    .agg(F.count("*").alias("InstallmentCount"))
    .filter(F.col("InstallmentCount") >= 12)
    .select("LoanAccount")
    .dropDuplicates()
)

GXLOneHolding_NonDenounced_Accounts_df.createOrReplaceTempView("GXLOneHolding_NonDenounced_Accounts")


# =====================================================================================
# Denounced / Non-Denounced classification
# =====================================================================================

bauexlusion_retail_GXLOneHolding_Final_df = (
    final_export_df_filtered_glx.alias("fe")
    .join(
        GXLOneHolding_NonDenounced_Accounts_df.alias("gna"),
        F.col("fe.LoanAccount") == F.col("gna.LoanAccount"),
        how="left"
    )
    .select(
        "fe.LoanAccount",
        F.when(F.col("gna.LoanAccount").isNotNull(), "Non-Denounced")
         .otherwise("Denounced")
         .alias("Account_is_Denounced_NonDenounced")
    )
    .dropDuplicates()
)

bauexlusion_retail_GXLOneHolding_Final_df.createOrReplaceTempView("bauexlusion_retail_GXLOneHolding_Final")

In [0]:
# =====================================================================================
# GLX One Iris DAC portfolio–specific enrichment logic for Final Export.
#
# Objective:
# 1) Identify GLX One Iris DAC accounts with:
#       - GBV > 0
#       - DenouncedDate IS NULL
#       - At least 12 future ACTIVE installments
#    Classify them as "Non-Denounced", others as "Denounced".
#
# 2) Prepare for CustomerPayerSegment (optional, similar to Cosmos/Galaxy/GXL One Holding)
#
# Output:
#   - Account-level Denounced / Non-Denounced flag
#
# All logic strictly applies to:
#   PortfolioName = 'GLX One Iris DAC'
# =====================================================================================

final_export_df_filtered_glx = (
    final_export_df
    .filter(F.col("PortfolioName") == "GLX One Iris DAC")
)

dexposure_filtered = (
    dexposure_profits
        .filter(
            (F.col("GBV") > 0) &
            (F.col("DenouncedDate").isNull())
        )
        .select("ExposureKey")
)


# =====================================================================================
# Retrieve all GLX One Iris DAC Loan Accounts
# =====================================================================================

GLXOneIrisDAC_Accounts_df = (
    final_export_df_filtered_glx
    .select("LoanAccount")
    .dropDuplicates()
)

GLXOneIrisDAC_Accounts_df.createOrReplaceTempView("GLXOneIrisDAC_Accounts")


# =====================================================================================
# Keep accounts satisfying GBV & DenouncedDate conditions
# =====================================================================================

GLXOneIrisDAC_Accounts_GBVpositive_DenounceDateNULL_df = (
    GLXOneIrisDAC_Accounts_df.alias("giris")
    .join(dexposure_filtered.alias("dex"), 
          F.col("giris.LoanAccount") == F.col("dex.ExposureKey"), 
          how="inner")
    .select("giris.LoanAccount")
    .dropDuplicates()
)

GLXOneIrisDAC_Accounts_GBVpositive_DenounceDateNULL_df.createOrReplaceTempView("GLXOneIrisDAC_Accounts_GBVpositive_DenounceDateNULL")


# =====================================================================================
# Retrieve future installments (InstallmentDate > BusinessDate, InstallmentAmount ≠ 0)
# Join with dexposure, that only has active installments, instead of dpaymentschedule
# =====================================================================================

GLXOneIrisDAC_Accounts_GBVpositive_DenounceDateNULL_AllActiveInstallements_df = (
    GLXOneIrisDAC_Accounts_GBVpositive_DenounceDateNULL_df.alias("oagd")
    .join(dexposure.alias("dex"), 
          F.col("oagd.LoanAccount") == F.col("dex.ExposureKey"),
          how="inner")
    .filter((F.col("dex.NextInstallmentDate") > F.col("dex.BusinessDate")) &
            (F.col("dex.NextInstallmentAmount") != 0)
            )
    .select("oagd.LoanAccount", "dex.NextInstallmentDate")
    .dropDuplicates()
)

GLXOneIrisDAC_Accounts_GBVpositive_DenounceDateNULL_AllActiveInstallements_df.createOrReplaceTempView("GLXOneIrisDAC_Accounts_GBVpositive_DenounceDateNULL_AllActiveInstallements")


# =====================================================================================
# Keep only ACTIVE installments
# =====================================================================================

GLXOneIrisDAC_NonDenounced_Accounts_df = (
    GLXOneIrisDAC_Accounts_GBVpositive_DenounceDateNULL_AllActiveInstallements_df
    .groupBy("LoanAccount")
    .agg(F.count("*").alias("InstallmentCount"))
    .filter(F.col("InstallmentCount") >= 12)
    .select("LoanAccount")
    .dropDuplicates()
)

GLXOneIrisDAC_NonDenounced_Accounts_df.createOrReplaceTempView("GLXOneIrisDAC_NonDenounced_Accounts")


# =====================================================================================
# Identify accounts with ≥ 12 active future installments
# =====================================================================================

bauexlusion_retail_GLXOneIrisDAC_Final_df = (
    final_export_df_filtered_glx.alias("fe")
    .join(
        GLXOneIrisDAC_NonDenounced_Accounts_df.alias("gna"),
        F.col("fe.LoanAccount") == F.col("gna.LoanAccount"),
        how="left"
    )
    .select(
        "fe.LoanAccount",
        F.when(F.col("gna.LoanAccount").isNotNull(), "Non-Denounced")
         .otherwise("Denounced")
         .alias("Account_is_Denounced_NonDenounced")
    )
    .dropDuplicates()
)

bauexlusion_retail_GLXOneIrisDAC_Final_df.createOrReplaceTempView("bauexlusion_retail_GLXOneIrisDAC_Final")

In [0]:
# =====================================================================================
# Combine the final account-level Denounced/Non-Denounced flags
# from all individual portfolios into a single table / DataFrame.
#
# Portfolios included: Orion, Galaxy II, Cosmos Retail, 
#                      GXL One Holding, GLX One Iris DAC
# =====================================================================================

bauexlusion_retail_final_df = (
    bauexlusion_retail_Orion_Final_df
    .unionByName(bauexlusion_retail_GalaxyII_Final_df)
    .unionByName(bauexlusion_retail_CosmosRetail_Final_df)
    .unionByName(bauexlusion_retail_GXLOneHolding_Final_df)
    .unionByName(bauexlusion_retail_GLXOneIrisDAC_Final_df)
    .unionByName(bauexlusion_retail_MONZARETAIL_Final_df)
)

bauexlusion_retail_final_df.createOrReplaceTempView("bauexlusion_retail_final")


# =====================================================================================
# Combine the CustomerPayerSegment per (LoanAccount, CDI) 
#  from the individual portfolios into a single DataFrame / table.
#
# Portfolios included: Orion, Galaxy II, Cosmos Retail
# =====================================================================================

CustomerPayerSegment_Retail_Final_df = (
    CustomerPayerSegment_Orion_df
    .unionByName(CustomerPayerSegment_GalaxyII_df)
    .unionByName(CustomerPayerSegment_CosmosRetail_df)
)

CustomerPayerSegment_Retail_Final_df.createOrReplaceTempView("CustomerPayerSegment_Retail_Final")

In [0]:

# =====================================================================================
# Wholesale portfolios Impact Marking Flag logic 
#
# Objective:
# 1) Determine which wholesale loan accounts are in an "active adjustment"
#    (Restructured or Settlement) for portfolios excluding 'Symbol'.
#
# 2) Identify accounts for all wholesale portfolios with DaysPastDue (dpd) < 90.
#
# Portfolios:
#   - Symbol (special case: only dpd < 90)
#   - Neptune
#   - Cosmos Wholesale
#   - Galaxy IV Funding DAC
#   - Hermes (Portfolios A, B, C, Common Customers)
#
# Logic summary:
#   a) Active adjustment via Modifications_PEE.IsRestructuredFlag = 1 
#      AND LoanStatus in ('Overdue','Normal') 
#      AND at least 1 future installment with InstallmentAmount <> 0
#   b) Active adjustment via Modifications_PEE.IsSettlementFlag = 1 
#      AND SettlementStatus in ('Running','Implemented')
#   c) For all accounts (including Symbol), dpd < 90
# =====================================================================================
                 
dpaymentschedule = spark.table("dpaymentschedule")  
settlementexposure = spark.table("settlementexposure")
dexposure_PEE_filtered = (
    dexposure
    .filter(
        (F.col("PEERestructureTypeCode").isNotNull()) &
        (F.col("HasActiveSettlementFlag").isNotNull()))
)


# =====================================================================================
# Wholesale accounts: DISTINCT LoanAccount and PortfolioName
# =====================================================================================

Wholesale_Accounts_df = (
    final_export_df
    .select(F.col("LoanAccount").alias("LoanAccount"), F.col("PortfolioName"))
    .dropDuplicates()   
)

Wholesale_Accounts_df.createOrReplaceTempView("Wholesale_Accounts")


# =====================================================================================
# Exclude Symbol for active adjustment check
# =====================================================================================

Wholesale_Accounts_Except_Symbol_df = (
    Wholesale_Accounts_df
    .filter(F.col("PortfolioName") != "Symbol")
    .select("LoanAccount")
    .dropDuplicates()
)

Wholesale_Accounts_Except_Symbol_df.createOrReplaceTempView("Wholesale_Accounts_Except_Symbol")


# =====================================================================================
# Active adjustments via RestructuredFlag
#  - LoanStatus in ('Overdue','Normal')
#  - Has at least one future installment
# =====================================================================================

Wholesale_NotSYM_RecturedFlag1_df = (
    Wholesale_Accounts_Except_Symbol_df.alias("waes")
    .join(dexposure_PEE_filtered.alias("dex"),
          F.col("waes.LoanAccount") == F.col("dex.ExposureKey"),
          how="inner")
    .select(F.col("waes.LoanAccount"))
    .dropDuplicates()
)

Wholesale_NotSYM_RecturedFlag1_df.createOrReplaceTempView("Wholesale_NotSYM_RecturedFlag1")


Wholesale_NotSYM_RecturedFlag1_OverDue_Normal_df = (
    Wholesale_NotSYM_RecturedFlag1_df.alias("wrf1")
    .join(dexposure.alias("dex"),
          F.col("wrf1.LoanAccount") == F.col("dex.ExposureKey"),
          how="inner")
    .filter(F.col("dex.ExposureStatus").isin("Overdue", "Normal"))
    .select(F.col("wrf1.LoanAccount"))
    .dropDuplicates()
)

Wholesale_NotSYM_RecturedFlag1_OverDue_Normal_df.createOrReplaceTempView("Wholesale_NotSYM_RecturedFlag1_OverDue_Normal")


Wholesale_NotSYM_RecturedFlag1_OverDue_Normal_Installments_df = (
    Wholesale_NotSYM_RecturedFlag1_OverDue_Normal_df.alias("wrf1on")
    .join(dexposure_profits.alias("dex"), 
          F.col("wrf1on.LoanAccount") == F.col("dex.ExposureKey"), 
          how="inner")
    .join(dpaymentschedule.alias("ps"), 
          F.col("ps.ExposureKey") == F.col("dex.ExposureKey"), 
          how="inner")
    .select(
        F.col("wrf1on.LoanAccount").alias("LoanAccount"),
        F.col("ps.ExposureKey").alias("ExposureKey"),
        F.col("ps.InstallmentAmount"),
        F.col("ps.InstallmentDate"),
        F.col("dex.BusinessDate")
    )
    .dropDuplicates()   
)

Wholesale_NotSYM_RecturedFlag1_OverDue_Normal_Installments_df.createOrReplaceTempView("Wholesale_NotSYM_RecturedFlag1_OverDue_Normal_Installments")


Wholesale_NotSYM_RecturedFlag1_OverDue_Normal_HasFutureInstallments_df = (
    Wholesale_NotSYM_RecturedFlag1_OverDue_Normal_Installments_df
    .filter(
        (F.col("InstallmentAmount").isNotNull()) & (F.col("InstallmentAmount") != 0) &
        (F.col("InstallmentDate") > F.col("BusinessDate"))
    )
    .select("LoanAccount")
    .dropDuplicates()
)

Wholesale_NotSYM_RecturedFlag1_OverDue_Normal_HasFutureInstallments_df.createOrReplaceTempView("Wholesale_NotSYM_RecturedFlag1_OverDue_Normal_HasFutureInstallments")


# =====================================================================================
# Active adjustments via SettlementFlag
# =====================================================================================

Wholesale_NotSYM_SettlementFlag1_df = (
    Wholesale_Accounts_Except_Symbol_df.alias("waes")
    .join(dexposure_PEE_filtered.alias("dex"), 
          F.col("waes.LoanAccount") == F.col("dex.ExposureKey"), 
          how="inner")
    .select(F.col("waes.LoanAccount"))
    .dropDuplicates()
)

Wholesale_NotSYM_SettlementFlag1_df.createOrReplaceTempView("Wholesale_NotSYM_SettlementFlag1")


Wholesale_NotSYM_SettlementFlag1_Settlement_Running_Implemented_df = (
    Wholesale_NotSYM_SettlementFlag1_df.alias("wsf1")
    .join(dexposure.alias("dex"), 
          F.col("wsf1.LoanAccount") == F.col("dex.ExposureKey"), 
          how="inner")
    .join(settlementexposure.alias("sexp"), 
          F.col("dex.ExposureKey") == F.col("sexp.exposureKey"), 
          how="inner")
    .filter(F.col("sexp.SettlementStatus").isin("Running", "Implemented"))
    .select(F.col("wsf1.LoanAccount"))
    .dropDuplicates()
)

Wholesale_NotSYM_SettlementFlag1_Settlement_Running_Implemented_df.createOrReplaceTempView("Wholesale_NotSYM_SettlementFlag1_Settlement_Running_Implemented")


Wholesale_NotSYM_HasActiveAdjustment_df = (
    Wholesale_NotSYM_RecturedFlag1_OverDue_Normal_HasFutureInstallments_df
    .select("LoanAccount")
    .unionByName(Wholesale_NotSYM_SettlementFlag1_Settlement_Running_Implemented_df
                 .select("LoanAccount")
                 )
    .dropDuplicates()
)

Wholesale_NotSYM_HasActiveAdjustment_df.createOrReplaceTempView("Wholesale_NotSYM_HasActiveAdjustment")


# =====================================================================================
# Accounts with DPD < 90 for all Wholesale portfolios (including Symbol)
# =====================================================================================

PastDueDays_less_than_90_df = (
    Wholesale_Accounts_df.alias("wa")
    .join(dexposure_profits.alias("dex"), 
          F.col("wa.LoanAccount") == F.col("dex.ExposureKey"), 
          how="inner"
          )
    .filter((F.col("dex.DaysPastDue") < 90))
    .select(F.col("wa.LoanAccount"), 
            F.col("dex.DaysPastDue")
            )
    .dropDuplicates()
)
    
PastDueDays_less_than_90_df.createOrReplaceTempView("PastDueDays_less_than_90")


In [0]:
# =====================================================================================
# Creation of [Impact Marking Flag] for Retail Portfolios
#
# Objective:
#   For all Retail portfolios (Orion, Galaxy II, Cosmos Retail, GXL One Holding,
#   GLX One Iris DAC), create the Impact Marking Flag at LoanAccount level.
#
# Business Rule:
#   If [Account_is_Denounced_NonDenounced] = 'Non-Denounced' → Impact Marking Flag = 'Yes'
#   Else → Impact Marking Flag = 'No'
#
# Output:
#   LoanAccount
#   Impact_Marking_Flag
# =====================================================================================

ImpactMarkingFlag_retail_df = (
    bauexlusion_retail_final_df
    .select(
        "LoanAccount",
        F.when(F.col("Account_is_Denounced_NonDenounced") == "Non-Denounced", "Yes")
         .otherwise("No")
         .alias("Impact_Marking_Flag")
    )
    .dropDuplicates()  
)

ImpactMarkingFlag_retail_df.createOrReplaceTempView("ImpactMarkingFlag_retail")


# =====================================================================================
# Creation of [Impact Marking Flag] for Wholesale Portfolios
#
# Portfolios: Symbol, Neptune, Cosmos Wholesale, Galaxy IV Funding DAC, Hermes Portfolios
#
# Business Rules:
# For portfolios EXCEPT 'Symbol':
#   If LoanAccount:
#       - Is in active adjustment (Restructured OR Settlement) AND
#       - DaysPastDue < 90
#   → Impact Marking Flag = 'Yes'
#
# For portfolio 'Symbol':
#   If DaysPastDue < 90 → Impact Marking Flag = 'Yes'
#   Else: → Impact Marking Flag = 'No'
#
# Output:
#   LoanAccount
#   PortfolioName
#   Impact_Marking_Flag
# =====================================================================================

ImpactMarkingFlag_wholesale_df = (
    Wholesale_Accounts_df.alias("wa")
    .join(Wholesale_NotSYM_HasActiveAdjustment_df.alias("wnshaa"),
          F.col("wa.LoanAccount") == F.col("wnshaa.LoanAccount"),
          how="left")
    .join(PastDueDays_less_than_90_df.alias("dpdl90"),
          F.col("wa.LoanAccount") == F.col("dpdl90.LoanAccount"),
          how="left")
    .select(
        F.col("wa.LoanAccount"),
        F.col("wa.PortfolioName"),
        F.when(
            (F.col("wnshaa.LoanAccount").isNotNull() & 
             F.col("dpdl90.LoanAccount").isNotNull()), 
            "Yes"
        )
        .when(
            (F.col("wnshaa.LoanAccount").isNull() & 
            (F.col("wa.PortfolioName") == "Symbol") & 
            F.col("dpdl90.LoanAccount").isNotNull()), 
            "Yes"
        )
        .otherwise("No")
        .alias("Impact_Marking_Flag")
    )
    .dropDuplicates() 
)

ImpactMarkingFlag_wholesale_df.createOrReplaceTempView("ImpactMarkingFlag_wholesale")

In [0]:
# =====================================================================================
# The Objective is to create the GroupInsurance as a base
# for the two output tables 
#
# Conditional selection of Impact_Marking_Flag:
#   If Portfolio in Retail list → use retail flag
#   If Portfolio in Wholesale list → use wholesale flag
# =====================================================================================

cust = spark.table("Customer").alias("cust")
cust1 = spark.table("Customer1").alias("cust1")
cc = spark.table("casecode").alias("cc")
imr = spark.table("ImpactMarkingFlag_retail").alias("imr")
imw = spark.table("ImpactMarkingFlag_wholesale").alias("imw")
cpsr = spark.table("CustomerPayerSegment_Retail_Final").alias("cpsr")


GroupInsurance_df = (
    final_export_df.alias("fe")
    .filter(F.coalesce(F.col("fe.policy_number_in"), F.lit("")) != "")
    .join(cust, 
          F.col("cust.cepal_key") == F.col("fe.cepal_key"), 
          how="left")
    .join(
        cust1,
        (F.col("cust1.cepal_key") == F.col("fe.cepal_key")) &
        (F.col("cust1.agreement_key") == F.col("fe.agreement_key")),
        how="left"
    )
    .join(
        cc,
        (F.col("cc.cepal_key") == F.col("fe.cepal_key")) &
        (F.col("cc.LoanAccount") == F.col("fe.LoanAccount")),
        how="left"
    )
    .join(imr, 
          F.col("imr.LoanAccount") == F.col("fe.LoanAccount"), 
          how="left"
    )
    .join(imw, 
          F.col("imw.LoanAccount") == F.col("fe.LoanAccount"), 
          how="left"
    )
    .join(
        cpsr,
        (F.col("cpsr.LoanAccount") == F.col("fe.LoanAccount")) &
        (F.col("cpsr.cdi") == F.col("fe.cdi")),
        how="left"
    )
    .select(
        F.col("fe.cepal_key"),
        F.col("fe.Property_Active_flag"),
        F.col("fe.collateral_status"),
        F.col("fe.LoanAccount"),
        F.col("fe.net_insurance_amount"),
        F.col("fe.forecast_insurance_amount"),
        F.col("fe.OutstandingTotalBalanceLoan"),
        F.col("fe.LoanStatus"),
        F.col("cust1.AFM"),
        F.col("cust1.cdi"),
        F.col("cust1.BorrowerName"),
        F.col("fe.relation_type"),
        F.col("cust.GroupCode"),
        F.col("cust.GroupName"),
        F.col("fe.insurance_group"),
        F.col("fe.insurance_company"),
        F.col("fe.agreement_key"),
        F.col("fe.ProductCode_profits"),
        F.col("fe.ProductDescription_profits"),
        F.col("fe.SubProductDescription_profits"),
        F.col("fe.policy_number"),
        F.col("fe.policy_number_in"),
        F.col("fe.insurance_start_date"),
        F.col("fe.insurance_end_date"),
        F.col("fe.NextInstallmentDate"),
        F.col("fe.NextInstallment"),
        F.col("fe.PortfolioName"),
        F.col("fe.solo_policy_number"),
        F.col("fe.solo_insurance_start_date"),
        F.col("fe.solo_insurance_end_date"),
        F.col("cust.Address"),
        F.col("cust.PostalCode"),
        F.col("cust.County"),
        F.col("fe.Loan_Selection_Flag"),
        F.col("fe.solo_insurance_received_date"),
        F.col("fe.solo_insurance_entered_date"),
        F.col("cc.CaseCode"),
        F.col("cc.TransactionFlag"),
        F.col("cpsr.CustomerPayerSegment"),
        F.when(
            F.col("fe.PortfolioName").isin(
                "Orion X Securitisation DAC", "Galaxy II Funding DAC", "Cosmos Retail", 
                "GXL One Holding", "GLX One Iris DAC",
                "MONZA PORTFOLIO - RETAIL"
            ),
            F.col("imr.Impact_Marking_Flag")
        ).when(
            F.col("fe.PortfolioName").isin(
                "Symbol", "Neptune", "Cosmos Wholesale", "Galaxy IV Funding DAC",
                "Hermes Portfolio A", "Hermes Portfolio B", "Hermes Common Customers", "Hermes Portfolio C",
                "EUCLID", "MONZA PORTFOLIO - WHOLESALE"
            ),
            F.col("imw.Impact_Marking_Flag")
        ).otherwise(F.lit(None)).alias("Impact_Marking_Flag"),
        F.col("cust1.L3869_Flag"),
        F.col("fe.BusinessDate")
    )
    .dropDuplicates()
)

GroupInsurance_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"`{env}-cdh-delivery`.internalreports.ip_GroupInsurance")
GroupInsurance_df = spark.table(f"`{env}-cdh-delivery`.internalreports.ip_GroupInsurance")

In [0]:
# =====================================================================================
# Creation of PropertiesGroupPremiumsD
#
# Exactly the same as GroupInsurance, detailed level ("D" = Detailed),
# with renamed columns for reporting/SSRS purposes 
# =====================================================================================

PropertiesGroupPremiumsD= (
    GroupInsurance_df
    .select(
        F.col("cepal_key"),
        F.col("Property_Active_flag"),
        F.col("collateral_status").alias("Collateral_re Status"),
        F.col("LoanAccount").alias("Loan Account"),
        F.col("net_insurance_amount").cast("decimal(30,4)").alias("Καθαρό Ασφάλιστρο"),
        F.col("forecast_insurance_amount").cast("decimal(19,4)").alias("Πρόβλεψη Ασφαλίστρου"),
        F.col("OutstandingTotalBalanceLoan"),
        F.col("LoanStatus"),
        F.col("AFM").alias("ΑΦΜ"),
        F.col("cdi").alias("crs"),
        F.col("BorrowerName").alias("ΟΝΟΜΑΤΕΠΩΝΥΜΟ ΟΦΕΙΛΕΤΗ / ΕΠΩΝΥΜΙΑ ΟΦΕΙΛΕΤΗ"),
        F.col("GroupCode"),
        F.col("GroupName"),
        F.col("insurance_group").alias("Insurance Group"),
        F.col("insurance_company").alias("Insurance Company"),
        F.col("agreement_key").alias("Αριθμός Σύμβασης"),
        F.col("ProductCode_profits").alias("Κωδικός Προϊόντος profits"),
        F.col("ProductDescription_profits").alias("Περιγραφή Προϊόντος profits"),
        F.col("SubProductDescription_profits").alias("Περιγραφή Sub Προϊόντος profits"),
        F.col("policy_number").alias("Αρ Ασφ Συμβολαίου"),
        F.col("policy_number_in").alias("Αρ Ασφ Συμβολαίου In"),
        F.col("insurance_start_date").alias("Ημερομηνία Έναρξης Ασφ"),
        F.col("insurance_end_date").alias("Ημερομηνία λήξης Ασφ"),
        F.col("NextInstallmentDate"),
        F.col("NextInstallment"),
        F.col("PortfolioName"),
        F.col("solo_policy_number").alias("Αρ Ατομικού Ασφ Συμβολαίου"),
        F.col("solo_insurance_start_date").alias("Ημερομηνία Έναρξης Ατομικού Ασφ"),
        F.col("solo_insurance_end_date").alias("Ημερομηνία Λήξης Ατομικού Ασφ"),
        F.col("Address").alias("Διέυθυνση"),
        F.col("PostalCode").alias("ΤΚ"),
        F.col("County").alias("Περιοχή"),
        F.col("Loan_Selection_Flag").alias("Loan Selection Flag"),
        F.col("solo_insurance_received_date").alias("Ημερομηνία Παραλαβής Ατομικού Ασφ"),
        F.col("solo_insurance_entered_date").alias("Ημερομηνία Εισαγωγής Ατομικού Ασφ"),
        F.col("CaseCode"),
        F.col("TransactionFlag"),
        F.col("CustomerPayerSegment"),
        F.col("Impact_Marking_Flag").alias("Impact Marking Flag"),
        F.col("L3869_Flag").alias("L3869 Flag"),
        F.col("BusinessDate")
    )
    .dropDuplicates()
)

PropertiesGroupPremiumsD.write.format("delta")\
    .option("delta.columnMapping.mode", "name") \
    .option("delta.minReaderVersion", "2") \
    .option("delta.minWriterVersion", "5") \
    .option("encoding", "UTF-8") \
    .option("overwriteSchema", "true") \
    .mode("overwrite") \
    .option("replaceWhere", f"BusinessDate = '{run_date}'") \
    .saveAsTable(f"`{env}-cdh-delivery`.internalreports.PropertiesGroupPremiumsD") 


# =====================================================================================
# Creation of PropertiesGroupPremiumsT
#
# Aggregated insurance premiums per property ("T" = Totalized),
# summing net/forecast premiums across ALL insurance contracts per cepal_key,
# min start date, max end date, joined with the most recent GroupInsurance record
# to keep other relevant details. One row per cepal_key.
# =====================================================================================

agg_df = (
    perimeter_final_df
    .select(
        "cepal_key",
        "policy_number",
        "policy_number_in",
        "net_insurance_amount",
        "forecast_insurance_amount",
        "insurance_start_date",
        "insurance_end_date"
    )
    .dropDuplicates()
    .groupBy("cepal_key", "policy_number_in")
    .agg(
        F.sum("net_insurance_amount").cast("decimal(30,4)").alias("net_insurance_amount"),   
        F.sum("forecast_insurance_amount").cast("decimal(19,4)").alias("forecast_insurance_amount"),   
        F.min(F.col("insurance_start_date").cast("date")).alias("insurance_start_date"),   
        F.max(F.col("insurance_end_date").cast("date")).alias("insurance_end_date")   
    )
)

# Dedup GroupInsurance: one row per cepal_key — pick the most recent contract
w_gi_dedup = Window.partitionBy("cepal_key").orderBy(
    F.col("insurance_end_date").desc_nulls_last(),
    F.col("insurance_start_date").desc_nulls_last()
)
gi_deduped = (
    GroupInsurance_df
    .withColumn("_gi_rnk", F.row_number().over(w_gi_dedup))
    .filter(F.col("_gi_rnk") == 1)
    .drop("_gi_rnk")
)

gi = gi_deduped.alias("gi")
ag = agg_df.alias("ag")

PropertiesGroupPremiumsT = (
    gi
    .join(
        ag,
        gi.cepal_key == ag.cepal_key,
        how="inner"
    )
    .select(
        F.col("gi.cepal_key"),
        F.col("gi.Property_Active_flag"),
        F.col("gi.collateral_status").alias("Collateral_re Status"),
        F.col("gi.LoanAccount").alias("Loan Account"),
        F.col("ag.net_insurance_amount").alias("Καθαρό Ασφάλιστρο"),
        F.col("ag.forecast_insurance_amount").alias("Πρόβλεψη Ασφαλίστρου"),
        F.col("gi.OutstandingTotalBalanceLoan"),
        F.col("gi.LoanStatus"),
        F.col("gi.AFM").alias("ΑΦΜ"),
        F.col("gi.cdi").alias("crs"),
        F.col("gi.BorrowerName").alias("ΟΝΟΜΑΤΕΠΩΝΥΜΟ ΟΦΕΙΛΕΤΗ / ΕΠΩΝΥΜΙΑ ΟΦΕΙΛΕΤΗ"),
        F.col("gi.GroupCode"),
        F.col("gi.GroupName"),
        F.col("gi.insurance_group").alias("Insurance Group"),
        F.col("gi.insurance_company").alias("Insurance Company"),
        F.col("gi.agreement_key").alias("Αριθμός Σύμβασης"),
        F.col("gi.ProductCode_profits").alias("Κωδικός Προϊόντος profits"),
        F.col("gi.ProductDescription_profits").alias("Περιγραφή Προϊόντος profits"),
        F.col("gi.SubProductDescription_profits").alias("Περιγραφή Sub Προϊόντος profits"),
        F.col("gi.policy_number_in").alias("Αρ Ασφ Συμβολαίου In"),
        F.col("ag.insurance_start_date"),
        F.col("ag.insurance_end_date"),
        F.col("gi.NextInstallmentDate"),
        F.col("gi.NextInstallment"),
        F.col("gi.PortfolioName"),
        F.col("gi.solo_policy_number").alias("Αρ Ατομικού Ασφ Συμβολαίου"),
        F.col("gi.solo_insurance_start_date").alias("Ημερομηνία Έναρξης Ατομικού Ασφ"),
        F.col("gi.solo_insurance_end_date").alias("Ημερομηνία Λήξης Ατομικού Ασφ"),
        F.col("gi.Address").alias("Διέυθυνση"),
        F.col("gi.PostalCode").alias("ΤΚ"),
        F.col("gi.County").alias("Περιοχή"),
        F.col("gi.Loan_Selection_Flag").alias("Loan Selection Flag"),
        F.col("gi.solo_insurance_received_date").alias("Ημερομηνία Παραλαβής Ατομικού Ασφ"),
        F.col("gi.solo_insurance_entered_date").alias("Ημερομηνία Εισαγωγής Ατομικού Ασφ"),
        F.col("gi.CaseCode"),
        F.col("gi.TransactionFlag"),
        F.col("gi.CustomerPayerSegment"),
        F.col("gi.Impact_Marking_Flag").alias("Impact Marking Flag"),
        F.col("gi.L3869_Flag").alias("L3869 Flag"),
        F.col("BusinessDate")
    )
    .dropDuplicates()
)

PropertiesGroupPremiumsT.write.format("delta")\
    .option("delta.columnMapping.mode", "name") \
    .option("delta.minReaderVersion", "2") \
    .option("delta.minWriterVersion", "5") \
    .option("encoding", "UTF-8") \
    .mode("overwrite") \
    .option("replaceWhere", f"BusinessDate = '{run_date}'") \
    .saveAsTable(f"`{env}-cdh-delivery`.internalreports.PropertiesGroupPremiumsT") 